# Guida Tecnica Completa a `gpt_model4.py`
## Un'Implementazione GPT-4 Style in PyTorch — Spiegazione Dettagliata in Italiano

---

> *Questa guida spiega ogni componente di `gpt_model4.py`: formule matematiche complete, esempi numerici, codice PyTorch eseguibile.*

## Struttura

| Cap. | Argomento |
|------|-----------|
| 0 | Prefazione e Contesto Storico |
| 1 | Panoramica di gpt_model4.py |
| 2 | Import e Dipendenze |
| 3 | RMSNorm |
| 4 | RoPE — Rotary Position Embedding |
| 5 | GroupedQueryAttention (GQA) |
| 6 | FeedForwardSwiGLU |
| 7 | TransformerBlock4 |
| 8 | GPT4Model |
| 9 | GPT4_CONFIG_SMALL |
| 10 | Confronto Architetturale |
| 11 | Diagramma del Flusso Dati |
| A-E | Appendici |


---
# Capitolo 0: Prefazione e Contesto Storico

## 0.1 L'Evoluzione dei Modelli Linguistici: RNN → LSTM → Transformer → GPT

La storia dei modelli linguistici neurali e' una storia di progressiva conquista della complessita' del linguaggio naturale. Per capire perche' `gpt_model4.py` e' costruito come lo e', dobbiamo ripercorrere l'evoluzione che ha portato all'architettura Transformer moderna.

### L'Era delle Reti Ricorrenti (2013-2017)

Prima dei Transformer, il paradigma dominante per il processamento del linguaggio erano le **Reti Neurali Ricorrenti (RNN)**. Una RNN processa una sequenza token per token, mantenendo uno **stato nascosto** h_t che accumula informazioni dal passato:

```
h_t = f(W_h * h_{t-1} + W_x * x_t + b)
```

Il problema fondamentale delle RNN e' il **vanishing gradient**: quando si propaga il gradiente attraverso molti timestep, esso tende a diventare esponenzialmente piccolo (o grande), rendendo difficile apprendere dipendenze a lungo raggio.

### LSTM: Long Short-Term Memory (Hochreiter & Schmidhuber, 1997)

Le LSTM introducono un **meccanismo di gate** esplicito per controllare il flusso di informazione:

```
f_t = sigma(W_f [h_{t-1}, x_t] + b_f)   # forget gate
i_t = sigma(W_i [h_{t-1}, x_t] + b_i)   # input gate
o_t = sigma(W_o [h_{t-1}, x_t] + b_o)   # output gate
C_t = f_t * C_{t-1} + i_t * tanh(W_C [h_{t-1}, x_t] + b_C)
```

Le LSTM mitigano il problema del vanishing gradient grazie alla **cella di memoria** C_t che puo' preservare informazioni a lungo termine. Tuttavia rimangono due limitazioni critiche:
1. **Sequenzialita'**: impossibile parallelizzare il training
2. **Contesto limitato**: anche con LSTM, le dipendenze molto lunghe rimangono difficili


## 0.2 'Attention Is All You Need' — La Rivoluzione del Transformer (2017)

Nel 2017, Vaswani et al. pubblicano il paper che cambiera' per sempre il machine learning: **'Attention Is All You Need'**. L'idea centrale: *eliminare completamente la ricorrenza* e basarsi **esclusivamente** sul meccanismo di attenzione.

### L'Architettura Originale del Transformer

Il Transformer originale e' un'architettura **encoder-decoder** per la traduzione automatica:

```
Input  --> [Encoder Stack] --> Encoder Output
Target --> [Decoder Stack] <-- (cross-attention)
        --> Output
```

**Encoder Block:**
```
x --> Multi-Head Self-Attention --> Add & Norm
  --> Feed-Forward Network      --> Add & Norm
```

### La Formula dell'Attenzione Scaled Dot-Product

Il meccanismo centrale:

```
Attention(Q, K, V) = softmax( Q @ K.T / sqrt(d_k) ) @ V
```

Dove:
- Q (query matrix): cosa stai cercando, shape (T, d_k)
- K (key matrix): cosa e' disponibile, shape (T, d_k)
- V (value matrix): cosa recuperi, shape (T, d_v)
- sqrt(d_k): fattore di scala per evitare saturazione del softmax

**Intuizione:** ogni posizione calcola un 'punteggio di compatibilita'' con tutte le altre posizioni (Q @ K.T), lo normalizza con softmax, e usa questi pesi per fare una media pesata dei valori V.

### Vantaggi rispetto a RNN/LSTM

| Proprieta' | RNN/LSTM | Transformer |
|------------|----------|-------------|
| Parallelismo | Nessuno (sequenziale) | Completo |
| Dipendenze a lungo raggio | Difficili | O(1) passi |
| Complessita' temporale | O(T) sequenziale | O(T^2) parallelizzabile |
| Training su GPU moderne | Lento | Molto efficiente |
| Memoria gradiente | Problematica | Highway diretto |


## 0.3 GPT-1, GPT-2, GPT-3, GPT-4: Evoluzione delle Scelte Architetturali

### GPT-1 (OpenAI, 2018)

'Improving Language Understanding by Generative Pre-Training' dimostra che il **pre-training non supervisionato** su grandi corpus, seguito da fine-tuning, produce risultati eccellenti.

- 117M parametri
- 12 layer Transformer **decoder-only** (solo self-attention causale)
- 768 dimensioni embedding, 12 teste di attenzione
- Positional embeddings **appresi** (learned absolute)
- Attivazione: GELU; Normalizzazione: LayerNorm (post-norm)

**Innovazione chiave:** Il decoder-only crea un modello **autoregressivo** che predice il token successivo dato il contesto. Semplice ma potente.

### GPT-2 (OpenAI, 2019)

'Language Models are Unsupervised Multitask Learners' dimostra che scalare il pre-training produce capacita' emergenti **senza** fine-tuning specifico.

| Variante | Parametri | Layer | Heads | Emb Dim |
|----------|-----------|-------|-------|--------|
| Small    | 124M      | 12    | 12    | 768    |
| Medium   | 355M      | 24    | 16    | 1024   |
| Large    | 774M      | 36    | 20    | 1280   |
| XL       | 1.5B      | 48    | 25    | 1600   |

**Modifiche rispetto a GPT-1:**
- Layer normalization spostata all'**inizio** di ogni sub-block (pre-norm)
- Layer normalization aggiunta dopo il blocco finale
- Inizializzazione dei pesi scalata per profondita' (divisione per sqrt(2*L))

### GPT-3 (OpenAI, 2020)

'Language Models are Few-Shot Learners' scala a **175 miliardi di parametri**, dimostrando capacita' di few-shot learning straordinarie. Stessa architettura di GPT-2 ma molto piu' grande.

### GPT-4 e i Moderni LLM (2023+)

OpenAI non ha pubblicato dettagli tecnici su GPT-4, ma attraverso paper contemporanei (LLaMA, PaLM-2, Mistral) si sono identificate 4 modifiche chiave rispetto a GPT-2:

1. **RMSNorm** invece di LayerNorm
2. **RoPE** invece di positional embeddings assoluti appresi
3. **Grouped Query Attention (GQA)** invece di Multi-Head Attention standard
4. **SwiGLU** invece di GELU nel Feed-Forward Network


## 0.4 LLaMA, Mistral, PaLM, GPT-NeoX: Convergenza verso le Stesse Tecniche

E' notevole che modelli sviluppati indipendentemente abbiano **convergito verso le stesse scelte**. Questo suggerisce che tali scelte riflettano proprieta' matematiche fondamentali.

### LLaMA (Meta, 2023)

- RMSNorm (pre-norm), RoPE, SwiGLU con hidden_dim = 8/3 * C
- Nessun bias nelle proiezioni lineari
- vocab_size = 32000 (SentencePiece tokenizer)

### LLaMA 2 (Meta, 2023)

- Aggiunge **Grouped Query Attention** per i modelli 34B e 70B
- Contesto esteso a 4096 token
- RLHF per i modelli chat

### LLaMA 3 (Meta, 2024)

- GQA anche nei modelli piccoli (8B)
- rope_base = **500,000** per supportare contesti molto lunghi
- Tokenizer: tiktoken con vocab_size = 128,256

### Mistral 7B (Mistral AI, 2023)

- GQA con n_heads=32, n_kv_heads=8
- Sliding Window Attention per ridurre la complessita'
- rope_base = 10000

### PaLM (Google, 2022)

- Multi-Query Attention (MQA: caso estremo di GQA con 1 solo KV head)
- Parallel layers: attention e FFN in parallelo (non in serie)
- SwiGLU

### Tabella Comparativa

| Modello | Norm | Pos. Enc. | Attenzione | FFN |
|---------|------|-----------|------------|-----|
| GPT-2 | LayerNorm post | Absolute learned | MHA | GELU 4x |
| LLaMA 1 | RMSNorm pre | RoPE | MHA | SwiGLU 8/3x |
| LLaMA 2 70B | RMSNorm pre | RoPE | GQA | SwiGLU 8/3x |
| Mistral 7B | RMSNorm pre | RoPE | GQA | SwiGLU |
| PaLM | LayerNorm pre | RoPE | MQA | SwiGLU |
| GPT-NeoX | LayerNorm pre | RoPE | MHA | GELU |
| **gpt_model4.py** | **RMSNorm pre** | **RoPE** | **GQA** | **SwiGLU** |


In [ ]:
# Capitolo 0: Confronto dei modelli nella storia
import math

modelli = [
    ('GPT-1 (2018)', 0.117, 'MHA+GELU+LayerNorm+AbsPos'),
    ('GPT-2 Small (2019)', 0.124, 'MHA+GELU+LayerNorm+AbsPos'),
    ('GPT-2 XL (2019)', 1.5, 'MHA+GELU+LayerNorm+AbsPos'),
    ('GPT-3 (2020)', 175.0, 'MHA+GELU+LayerNorm+AbsPos'),
    ('LLaMA 7B (2023)', 7.0, 'MHA+SwiGLU+RMSNorm+RoPE'),
    ('LLaMA 2 70B (2023)', 70.0, 'GQA+SwiGLU+RMSNorm+RoPE'),
    ('Mistral 7B (2023)', 7.0, 'GQA+SwiGLU+RMSNorm+RoPE'),
    ('LLaMA 3 8B (2024)', 8.0, 'GQA+SwiGLU+RMSNorm+RoPE'),
    ('gpt_model4.py (~GPT-2S)', 0.124, 'GQA+SwiGLU+RMSNorm+RoPE'),
]

print('=' * 70)
print(f'{"Modello":<30} {"Param":>8}  Architettura')
print('=' * 70)
for nome, mld, arch in modelli:
    print(f'{nome:<30} {mld:>6.3f}B  {arch}')
print('=' * 70)
print()
print('gpt_model4.py usa la stessa dimensione di GPT-2 Small')
print('ma con architettura modernizzata: RMSNorm, RoPE, GQA, SwiGLU')


## 0.5 Le 4 Modifiche Chiave: Panoramica

`gpt_model4.py` implementa precisamente le 4 modifiche che caratterizzano i moderni LLM:

### Modifica 1: RMSNorm (Capitolo 3)
- **Problema risolto:** LayerNorm calcola sia media che varianza -> costoso
- **Soluzione:** RMSNorm calcola solo la radice quadratica media (no centramento)
- **Guadagno:** ~15-30% piu' veloce su GPU, stessa qualita' empirica
- **Parametri risparmiati:** elimina il vettore beta (C parametri per layer)

### Modifica 2: RoPE — Rotary Position Embedding (Capitolo 4)
- **Problema risolto:** Positional embeddings assoluti non generalizzano a lunghezze out-of-distribution
- **Soluzione:** Ruotare Q e K in funzione della posizione, codificando la posizione **relativa**
- **Guadagno:** Migliore generalizzazione, estendibilita' del contesto modificando rope_base
- **Parametri risparmiati:** nessun pos_emb (T*C parametri eliminati)

### Modifica 3: Grouped Query Attention (Capitolo 5)
- **Problema risolto:** KV-cache enorme durante l'inferenza autoregressiva (O(T*H*D) per layer)
- **Soluzione:** Condividere KV heads tra gruppi di Q heads
- **Guadagno:** KV-cache ridotta di H/G volte (qui: 12/4 = 3x), quasi nessuna perdita di qualita'

### Modifica 4: SwiGLU (Capitolo 6)
- **Problema risolto:** GELU standard e' subottimale come non-linearita' nel FFN
- **Soluzione:** Gated linear unit con attivazione SiLU
- **Guadagno:** Migliore qualita' con lo stesso numero di parametri totali


---
# Capitolo 1: Panoramica di `gpt_model4.py`

## 1.1 Cosa Implementa il File

`gpt_model4.py` e' un'implementazione completa in PyTorch di un modello linguistico **GPT-4 style** (in dimensioni ridotte per scopi educativi). Il file definisce:

1. **`RMSNorm`** — Normalizzazione efficiente senza calcolo della media
2. **`precompute_rope_freqs()`** — Pre-calcolo delle frequenze per RoPE
3. **`apply_rope()`** — Applicazione della rotazione posizionale a Q e K
4. **`GroupedQueryAttention`** — Attenzione multi-testa con KV heads condivisi
5. **`FeedForwardSwiGLU`** — Rete feed-forward con gate SwiGLU
6. **`TransformerBlock4`** — Blocco Transformer completo (attenzione + FFN)
7. **`GPT4Model`** — Modello completo: embedding -> N blocchi -> output
8. **`GPT4_CONFIG_SMALL`** — Dizionario di configurazione default

Il file e' progettato per essere **didattico**: ogni riga ha commenti che spiegano le trasformazioni e le shape dei tensori.


## 1.2 Tabella di Confronto: GPT-2 vs GPT-4 Style

| Componente | GPT-2 (`gpt_model.py`) | GPT-4 Style (`gpt_model4.py`) |
|------------|------------------------|-------------------------------|
| **Normalizzazione** | `LayerNorm` (media + varianza) | `RMSNorm` (solo varianza) |
| **Posizione** | `pos_emb` appresi assoluti | `RoPE` (rotazione in attenzione) |
| **Attenzione** | MHA (H=12 Q,K,V heads) | GQA (H=12 Q, G=4 K/V heads) |
| **Feed-Forward** | GELU, 4xC hidden | SwiGLU, 8/3xC hidden |
| **Bias in QKV** | Si' | No (LLaMA style) |
| **Embedding posizionale** | `nn.Embedding(T, C)` | Nessuno (RoPE e' implicito) |
| **Parametri totali** | ~124M | ~124M (quasi identico) |
| **KV-cache (inferenza)** | H*D per layer | G*D per layer (3x riduzione) |

## 1.3 Diagramma dell'Architettura

```
Input: indici token (B, T)
         |
         v
   +-----------+
   |  tok_emb  |  nn.Embedding(50257, 768)
   |  drop_emb |  Dropout(0.1)
   +-----+-----+
         | (B, T, 768)
         |
   +-----+---------------------------+
   |  TransformerBlock4 x 12         |
   |  +---------------------------+  |
   |  |  norm1 (RMSNorm)          |  |
   |  |  GroupedQueryAttention    |<-RoPE
   |  |  drop_shortcut            |  |
   |  |  + shortcut (residual)    |  |
   |  |  norm2 (RMSNorm)          |  |
   |  |  FeedForwardSwiGLU        |  |
   |  |  drop_shortcut            |  |
   |  |  + shortcut (residual)    |  |
   |  +---------------------------+  |
   +-----+---------------------------+
         | (B, T, 768)
         v
   +-----------+
   | final_norm|  RMSNorm(768)
   |  out_head |  Linear(768, 50257)
   +-----+-----+
         | (B, T, 50257)
         v
      logits
```


In [3]:
# Capitolo 1: Ispezione dei componenti di gpt_model4.py
componenti = [
    ('Classe',   'RMSNorm',                 'righe 15-25',  'Normalizzazione RMS'),
    ('Funzione', 'precompute_rope_freqs()',  'righe 38-56',  'Pre-calcola cos/sin per RoPE'),
    ('Funzione', 'apply_rope()',             'righe 59-82',  'Applica rotazione RoPE a Q/K'),
    ('Classe',   'GroupedQueryAttention',    'righe 96-164', 'GQA con RoPE'),
    ('Classe',   'FeedForwardSwiGLU',        'righe 180-196','FFN con gate SwiGLU'),
    ('Classe',   'TransformerBlock4',        'righe 204-237','Blocco Transformer completo'),
    ('Classe',   'GPT4Model',                'righe 246-270','Modello completo'),
    ('Config',   'GPT4_CONFIG_SMALL',        'righe 277-287','Configurazione di default'),
]
print(f'{"Tipo":<10} {"Nome":<30} {"Righe":<15} Descrizione')
print('-' * 75)
for tipo, nome, righe, desc in componenti:
    print(f'{tipo:<10} {nome:<30} {righe:<15} {desc}')


Tipo       Nome                           Righe           Descrizione
---------------------------------------------------------------------------
Classe     RMSNorm                        righe 15-25     Normalizzazione RMS
Funzione   precompute_rope_freqs()        righe 38-56     Pre-calcola cos/sin per RoPE
Funzione   apply_rope()                   righe 59-82     Applica rotazione RoPE a Q/K
Classe     GroupedQueryAttention          righe 96-164    GQA con RoPE
Classe     FeedForwardSwiGLU              righe 180-196   FFN con gate SwiGLU
Classe     TransformerBlock4              righe 204-237   Blocco Transformer completo
Classe     GPT4Model                      righe 246-270   Modello completo
Config     GPT4_CONFIG_SMALL              righe 277-287   Configurazione di default


---
# Capitolo 2: Import e Dipendenze (Righe 1-5)

```python
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
```

## 2.1 `import math` — Il Modulo Standard Python

Il modulo `math` viene importato per **una sola operazione** nel file:

```python
hidden_dim = int(math.ceil((8 / 3 * emb_dim) / 256) * 256)
```

**Perche' `math.ceil` invece di `torch.ceil`?**

Il calcolo di `hidden_dim` avviene **a tempo di costruzione del modello** (`__init__`), non durante il forward pass. In questo momento non operiamo su tensori PyTorch ma su **scalari Python**. Usare `math.ceil` e':
- Piu' efficiente (nessun overhead di creazione tensore)
- Piu' leggibile (e' chiaramente un'operazione scalare)
- Senza dipendenze (non richiede CUDA o dispositivi specifici)

`math.ceil(x)` restituisce il piu' piccolo intero >= x:
- `math.ceil(2.3)` -> `3`
- `math.ceil(682.67)` -> `683`
- `math.ceil(256.0)` -> `256`


## 2.2 `import torch` — Il Framework di Deep Learning

PyTorch e' il framework di deep learning usato in tutto il file.

| Funzione torch | Uso in gpt_model4.py |
|----------------|----------------------|
| `torch.ones(n)` | Inizializzazione del parametro `scale` in RMSNorm |
| `torch.arange(n)` | Creazione degli indici in `precompute_rope_freqs` |
| `torch.outer(a, b)` | Prodotto esterno posizioni x frequenze per RoPE |
| `torch.cos(x)`, `torch.sin(x)` | Calcolo tavole look-up per RoPE |
| `torch.cat([...], dim=-1)` | Concatenazione in `apply_rope` |
| `torch.ones(T, T)` | Creazione della maschera causale |
| `torch.triu(M, diagonal=1)` | Triangolo superiore per maschera causale |
| `torch.softmax(x, dim=-1)` | Normalizzazione dei pesi di attenzione |
| `torch.inf` | Riempimento con -inf nella maschera causale |

### Tensori e Grafo Computazionale

PyTorch costruisce un **grafo computazionale dinamico** (define-by-run). Ogni operazione su tensori crea nodi nel grafo usati per calcolare i gradienti durante il `backward()`. La propagazione in avanti e' identica al codice Python normale — non ci sono declaration o compilazione preliminare.


## 2.3 `import torch.nn as nn` — Modulo delle Reti Neurali

`torch.nn` fornisce i building blocks per costruire reti neurali:

### `nn.Module` — La Classe Base

Ogni classe in `gpt_model4.py` eredita da `nn.Module`:
```python
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()  # OBBLIGATORIO
        ...
```

La chiamata a `super().__init__()` inizializza strutture interne:
- `_parameters`: dizionario dei tensori appresi
- `_buffers`: dizionario dei tensori non-appresi (es. maschera causale)
- `_modules`: dizionario dei sotto-moduli

### `nn.Parameter` — Tensori Appresi

```python
self.scale = nn.Parameter(torch.ones(emb_dim))
```

Quando chiami `model.parameters()`, `scale` viene incluso. Il gradiente viene automaticamente calcolato durante il backward.

### `nn.Linear(in, out, bias=...)` — Trasformazione Lineare

Implementa `y = x @ W.T + b` dove W ha shape `(out, in)`.
Usato per le proiezioni Q, K, V in `GroupedQueryAttention` e per le 3 matrici di `FeedForwardSwiGLU`.

### `nn.Embedding(num, dim)` — Tabella di Look-up

Shape dei pesi: `(num, dim)` = `(50257, 768)` per `tok_emb`.
Input: indici interi `(B, T)` -> Output: embedding `(B, T, dim)`.
Differenziabile: il gradiente si propaga solo alle righe usate.

### `nn.Dropout(p)` e `nn.Sequential`

- `nn.Dropout(p)`: azzera casualmente una frazione `p` degli elementi durante il training
- `nn.Sequential(*modules)`: applica i moduli in sequenza, usato per lo stack dei 12 blocchi


## 2.4 `import torch.nn.functional as F` — Operazioni Funzionali

`torch.nn.functional` contiene le stesse operazioni di `nn` ma come **funzioni pure** (senza stato).

In `gpt_model4.py` viene usato per:

| Funzione | Uso |
|----------|-----|
| `F.silu(x)` | Attivazione SiLU in `FeedForwardSwiGLU.forward()` |

**SiLU (Sigmoid Linear Unit):**
```
SiLU(x) = x * sigmoid(x) = x / (1 + exp(-x))
```

SiLU non ha parametri — e' una funzione pura. Per questo si usa `F.silu` nel `forward` invece di istanziare un modulo.

### Differenza tra `nn` e `F`

| `nn.Module` | `F` (funzionale) |
|-------------|------------------|
| Ha stato (parametri) | Senza stato |
| Es. `nn.Linear`, `nn.LayerNorm()` | Es. `F.linear()`, `F.layer_norm()` |
| Usato in `__init__` | Usato in `forward` |
| Gestisce training/eval automaticamente | Controllato manualmente |


In [ ]:
# Capitolo 2: Demo degli import e delle operazioni base
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

print('1. math.ceil per calcolo hidden_dim:')
for emb_dim in [256, 384, 512, 768, 1024]:
    raw = 8 / 3 * emb_dim
    hidden = int(math.ceil(raw / 256) * 256)
    print(f'   emb_dim={emb_dim:4d} -> raw={raw:8.2f} -> hidden_dim={hidden:4d}')

print()
print('2. torch.triu per maschera causale (T=5):')
mask = torch.triu(torch.ones(5, 5), diagonal=1)
print(mask)
print('  1=token futuro (mascherato con -inf), 0=token visibile')

print()
print('3. Funzioni di attivazione a confronto:')
x = torch.linspace(-3, 3, 7)
print(f'   x      = {x.numpy().round(2)}')
print(f'   ReLU   = {F.relu(x).numpy().round(3)}')
print(f'   GELU   = {F.gelu(x).numpy().round(3)}')
print(f'   SiLU   = {F.silu(x).numpy().round(3)}')

print()
print('4. nn.Parameter vs tensore normale:')
scale_param = nn.Parameter(torch.ones(4))
scale_tensor = torch.ones(4)
print(f'   nn.Parameter: requires_grad={scale_param.requires_grad}')
print(f'   torch.Tensor: requires_grad={scale_tensor.requires_grad}')


---
# Capitolo 3: RMSNorm — Root Mean Square Layer Normalization

## 3.1 Il Problema della Normalizzazione nelle Reti Neurali Profonde

Addestrare reti neurali profonde presenta un problema fondamentale: durante la propagazione in avanti, le distribuzioni delle attivazioni cambiano ad ogni layer. Questo fenomeno e' chiamato **Internal Covariate Shift** (Ioffe & Szegedy, 2015).

Senza normalizzazione:
- Le attivazioni possono **esplodere** (valori grandi -> saturazione di sigmoid/tanh)
- Le attivazioni possono **collassare** (valori ~0 -> gradienti che svaniscono)
- Il training richiede learning rate molto piccoli e warmup molto lungo
- Alta sensibilita' all'inizializzazione dei pesi

### Batch Normalization (Ioffe & Szegedy, 2015)

La prima soluzione proposta normalizza rispetto alla dimensione del batch:

```
x_hat_i = (x_i - mu_B) / sqrt(sigma_B^2 + eps)
```

Dove mu_B e sigma_B^2 sono calcolati sul batch. Problemi con BatchNorm nel linguaggio:
- Il batch per sequenze e' spesso piccolo (o 1 durante l'inferenza)
- La statistica del batch varia tra training e inferenza
- Non adatto per modelli autoregressivi


## 3.2 LayerNorm: Formula Matematica Completa

**Layer Normalization** (Ba et al., 2016) normalizza su ogni esempio individualmente, lungo la dimensione delle feature. Questo la rende indipendente dal batch.

Per un vettore x in R^C (un singolo token embedding di dimensione C):

**Passo 1: Calcola la media:**
```
mu = (1/C) * sum(x_i for i in range(C))
```

**Passo 2: Calcola la varianza:**
```
sigma^2 = (1/C) * sum((x_i - mu)^2 for i in range(C))
```

**Passo 3: Normalizza:**
```
x_hat_i = (x_i - mu) / sqrt(sigma^2 + eps)
```

**Passo 4: Riscala e trasla con parametri appresi:**
```
y_i = gamma_i * x_hat_i + beta_i
```

Dove:
- `gamma` in R^C: parametro di scala (inizializzato a 1)
- `beta` in R^C: parametro di bias/shift (inizializzato a 0)
- `eps`: costante numerica (tipicamente 1e-5)

**In GPT-2:**
```python
# GPT-2 usa LayerNorm:
self.norm = nn.LayerNorm(emb_dim)
# Parametri: gamma (C,) + beta (C,) = 2*C parametri per layer
```


## 3.3 RMSNorm: Derivazione Matematica (Zhang & Sennrich, 2019)

Zhang & Sennrich si chiedono: **e' davvero necessario sottrarre la media?**

La risposta empirica e' **no**, e questo porta a RMSNorm.

### Definizione Formale

Per un vettore x in R^C:

**RMS (Root Mean Square):**
```
RMS(x) = sqrt( (1/C) * sum(x_i^2) + eps )
```

**Normalizzazione:**
```
x_bar_i = x_i / RMS(x)
```

**Output con parametro di scala apprendibile gamma:**
```
y_i = gamma_i * x_bar_i = gamma_i * x_i / RMS(x)
```

### Differenze Fondamentali rispetto a LayerNorm

| Aspetto | LayerNorm | RMSNorm |
|---------|-----------|--------|
| Calcolo media mu | Si' | **No** |
| Denominatore | sqrt(Var(x) + eps) | sqrt(E[x^2] + eps) = RMS(x) |
| Centramento | Si' (sottrae mu) | **No** |
| Parametri | gamma + beta | Solo gamma |
| Parametri per layer (C=768) | 1536 | **768** |
| Operazioni (approx.) | ~7C | **~4C** |

### Equivalenza Asintotica

Se la media mu e' vicina a zero (comune nelle attivazioni dei Transformer dopo il warmup):

```
sigma^2 = E[(x - mu)^2] = E[x^2] - mu^2 ≈ E[x^2]  quando mu ≈ 0
=> LayerNorm ≈ RMSNorm
```

Nei Transformer con pre-norm, le attivazioni tendono naturalmente ad avere media vicina a zero, rendendo l'approssimazione accurata in pratica.


## 3.4 Implementazione: `class RMSNorm` — Riga per Riga

```python
class RMSNorm(nn.Module):  # righe 15-25
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
```

**`class RMSNorm(nn.Module):`**
Eredita da `nn.Module`. Ogni componente di rete neurale deve essere un Module per beneficiare di `parameters()`, `state_dict()`, `to(device)`, etc.

**`super().__init__()`**
Inizializza `nn.Module`, registrando i dizionari interni `_parameters`, `_buffers`, `_modules`. Senza questa chiamata, assegnare `nn.Parameter` solleverebbe un'eccezione.

**`self.eps = eps`**
Salva epsilon=1e-5 come attributo Python semplice (non un tensore). Aggiunto al denominatore per evitare divisione per zero quando RMS(x) ~= 0.

**`self.scale = nn.Parameter(torch.ones(emb_dim))`**
- `torch.ones(emb_dim)`: tensore shape `(C,)` = `(768,)` inizializzato a **1.0**
- `nn.Parameter(...)`: lo avvolge rendendolo visibile a `model.parameters()`
- Inizializzazione a 1 = identita' all'inizio del training
- Corrisponde a gamma nella formula matematica
- **Nessun `beta`**: RMSNorm non ha il parametro di bias

```python
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        norm_x = x / rms
        return self.scale * norm_x
```

**`rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)`**

Decomponendo passo per passo con x.shape = (B, T, C):
1. `x.pow(2)`: shape `(B, T, C)` -> `(B, T, C)` (quadrato elemento per elemento)
2. `.mean(dim=-1, keepdim=True)`: media lungo dim C, shape -> `(B, T, 1)`
3. `+ self.eps`: aggiunge epsilon per stabilita' numerica
4. `torch.sqrt(...)`: radice quadrata, shape -> `(B, T, 1)`

**`norm_x = x / rms`**
Broadcasting: `(B, T, C)` / `(B, T, 1)` -> `(B, T, C)`.
Ogni delle C componenti di ogni token viene diviso per lo stesso RMS scalare.

**`return self.scale * norm_x`**
Broadcasting: `(C,)` * `(B, T, C)` -> `(B, T, C)`.
Il parametro `scale` (gamma) moltiplica elemento per elemento lungo l'ultima dimensione.


In [ ]:
# Capitolo 3: Demo RMSNorm — implementazione e verifica
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(emb_dim))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        norm_x = x / rms
        return self.scale * norm_x

torch.manual_seed(42)
B, T, C = 2, 3, 8
x = torch.randn(B, T, C)

ln = nn.LayerNorm(C)
rn = RMSNorm(C)

print('Input x[0,0]:', x[0,0].detach().numpy().round(3))
print(f'  mean={x[0,0].mean():.4f}, std={x[0,0].std():.4f}')

with torch.no_grad():
    out_ln = ln(x)
    out_rn = rn(x)

print()
print('Output LayerNorm[0,0]:', out_ln[0,0].detach().numpy().round(3))
print(f'  mean={out_ln[0,0].mean():.4f} (garantito ~0)')

print()
print('Output RMSNorm[0,0]:', out_rn[0,0].detach().numpy().round(3))
rms_val = x[0,0].pow(2).mean().sqrt().item()
print(f'  RMS(x) = {rms_val:.4f}')
print(f'  mean={out_rn[0,0].mean():.4f} (non necessariamente 0)')

print()
print('Forme intermedie con x.shape=(2,3,8):')
print(f'  x.pow(2).shape          = {x.pow(2).shape}')
print(f'  .mean(-1,keepdim=True)  = {x.pow(2).mean(dim=-1,keepdim=True).shape}')
rms_tensor = torch.sqrt(x.pow(2).mean(dim=-1,keepdim=True)+1e-5)
print(f'  rms.shape               = {rms_tensor.shape}')
print(f'  (x/rms).shape           = {(x/rms_tensor).shape}')
print(f'  scale.shape             = {rn.scale.shape}')
print(f'  output.shape            = {out_rn.shape}')
print()
print('Parametri:')
print(f'  LayerNorm: {sum(p.numel() for p in ln.parameters())} (gamma + beta)')
print(f'  RMSNorm:   {sum(p.numel() for p in rn.parameters())} (solo gamma)')


In [ ]:
# Capitolo 3: Benchmark RMSNorm vs LayerNorm
import torch
import torch.nn as nn
import time

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return self.scale * (x / rms)

C = 768
B, T = 32, 512
x = torch.randn(B, T, C)
ln = nn.LayerNorm(C)
rn = RMSNorm(C)

# Warmup
for _ in range(20):
    _ = ln(x); _ = rn(x)

N = 500
t0 = time.perf_counter()
for _ in range(N):
    _ = ln(x)
t_ln = (time.perf_counter() - t0) / N * 1000

t0 = time.perf_counter()
for _ in range(N):
    _ = rn(x)
t_rn = (time.perf_counter() - t0) / N * 1000

print(f'Benchmark CPU (B={B}, T={T}, C={C}, N={N} iter):')
print(f'  LayerNorm: {t_ln:.3f} ms/iter')
print(f'  RMSNorm:   {t_rn:.3f} ms/iter')
speedup = (t_ln - t_rn) / t_ln * 100
direction = 'piu veloce' if speedup > 0 else 'piu lento'
print(f'  RMSNorm e {direction} del {abs(speedup):.1f}% su CPU')
print()
print('Su GPU A100: RMSNorm tipicamente ~15-30% piu veloce per C=768')
print('Il guadagno e proporzionale a C (piu grande C, piu si sente)')


## 3.5 Importanza per la Stabilita' del Training

### Pre-Norm Architecture

In `gpt_model4.py` (come in LLaMA), RMSNorm viene applicata **prima** di ogni sub-layer (pre-norm), non dopo (post-norm come nel Transformer originale).

**Post-norm (Vaswani 2017 originale):**
```
x -> Attention(x) -> x + Attention(x) -> LayerNorm -> output
```

**Pre-norm (GPT-2 in poi, incluso gpt_model4.py):**
```
x -> RMSNorm(x) -> Attention -> + x (residual) -> output
```

Con pre-norm:
- Il segnale residuale `x` fluisce **direttamente** senza normalizzazione
- Questo **garantisce il flusso del gradiente** attraverso le skip connections
- Training piu' stabile, meno sensibile all'inizializzazione
- Meno necessita' di warmup lungo

### Il Ruolo di `eps=1e-5`

Senza epsilon, se RMS(x) = 0 (es. token embedding tutto zero), si avrebbe una divisione per zero. `eps=1e-5` e' il valore standard:
- Abbastanza piccolo da non influenzare i calcoli normali (RMS ~= 1)
- Abbastanza grande da prevenire instabilita' numerica

### Dove Viene Usato in gpt_model4.py

RMSNorm appare in 3 contesti:
1. `self.norm1` in `TransformerBlock4`: prima di GQA
2. `self.norm2` in `TransformerBlock4`: prima di SwiGLU
3. `self.final_norm` in `GPT4Model`: prima della testa di output

Con 12 blocchi: 12*2 + 1 = **25 istanze di RMSNorm** nel modello completo.


---
# Capitolo 4: RoPE — Rotary Position Embedding

## 4.1 Il Problema dell'Encoding Posizionale

I Transformer non hanno un senso intrinseco dell'ordine sequenziale: l'attenzione e' una **funzione di insieme** (set function) — lo stesso risultato si otterrebbe con i token in ordine diverso. Dobbiamo quindi **iniettare informazioni posizionali** nel modello.

### Positional Embeddings Assoluti Appresi (GPT-2)

```python
# GPT-2:
self.pos_emb = nn.Embedding(context_length, emb_dim)
# Nel forward:
pos_embeds = self.pos_emb(torch.arange(T))
x = tok_embeds + pos_embeds  # addizione all'input
```

Ogni posizione 0, 1, ..., T-1 ha un vettore apprendo di dimensione C.

**Vantaggi:**
- Semplice da implementare
- Flessibile: il modello impara la rappresentazione ottimale

**Limitazioni:**
- **Out-of-distribution**: se la sequenza di test e' piu' lunga di quella di training, le posizioni oltre T non hanno embedding
- **Assoluto non relativo**: la posizione 5 non 'sa' che e' a distanza 2 dalla posizione 3
- **Non generalizza**: la posizione 500 in training non e' correlata alla posizione 500 in test
- **Parametri extra**: T*C = 1024*768 = ~786K parametri solo per le posizioni

### Sinusoidal Embeddings (Vaswani 2017)

Il paper originale usa embeddings fissi basati su seno e coseno:
```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Migliori per la generalizzazione ma ancora **assoluti**.


## 4.2 RoPE: Idea Centrale (Su et al., 2021)

**RoFormer: Enhanced Transformer with Rotary Position Embedding** propone un approccio radicalmente diverso: invece di aggiungere informazione posizionale all'input, **ruotiamo** i vettori Q e K all'interno di ogni testa di attenzione.

### L'Intuizione Matematica

Vogliamo che il dot product Q_p . K_q (tra query alla posizione p e key alla posizione q) dipenda **solo dalla distanza relativa** (p - q), non dalle posizioni assolute.

Se definiamo una funzione di rotazione R(pos) tale che:
```
R(p) @ R(q).T = R(p-q)
```

Allora il dot product diventa:
```
(R(p) @ q).T @ (R(q) @ k) = q.T @ R(p).T @ R(q) @ k
                             = q.T @ R(p-q) @ k
```

Che dipende solo da (p - q)!

### La Rotazione 2D

In 2D, ruotare un vettore [x1, x2] di un angolo theta:
```
[x1', x2'] = [x1*cos(theta) - x2*sin(theta),
               x2*cos(theta) + x1*sin(theta)]
```

In forma matriciale:
```
[cos(t)  -sin(t)] [x1]   [x1*cos(t) - x2*sin(t)]
[sin(t)   cos(t)] [x2] = [x2*cos(t) + x1*sin(t)]
```

### Da 2D a D Dimensioni

RoPE applica rotazioni **a coppie di dimensioni**. Per un vettore x di dimensione D:
- Coppia (0, 1): ruota di theta_0 * pos
- Coppia (2, 3): ruota di theta_1 * pos
- Coppia (2i, 2i+1): ruota di theta_i * pos

La matrice di rotazione totale e' **block-diagonale** con blocchi 2x2.


## 4.3 Le Frequenze theta_i

Le frequenze di rotazione per ogni coppia di dimensioni sono definite come:

```
theta_i = 1 / (base^(2i / D))    per i = 0, 1, ..., D/2 - 1
```

Con `base = 10000` (default) e `D = head_dim = 64`:

```
i=0:  theta_0 = 1/10000^(0/64)  = 1.0000  (alta frequenza)
i=1:  theta_1 = 1/10000^(2/64)  = 0.6607
i=4:  theta_4 = 1/10000^(8/64)  = 0.1930
i=16: theta_16= 1/10000^(32/64) = 0.0100
i=31: theta_31= 1/10000^(62/64) = 0.0015  (bassa frequenza)
```

### Analogia con l'Orologio

Pensate alle lancette di un orologio:
- **Lancetta dei secondi**: ruota velocemente (alta frequenza)
- **Lancetta dei minuti**: ruota lentamente (media frequenza)
- **Lancetta delle ore**: ruota molto lentamente (bassa frequenza)

Guardando tutte e tre le lancette, potete determinare il tempo esatto. RoPE fa la stessa cosa con le coppie di dimensioni del vettore.

### Spettro di Frequenze

- **Dimensioni basse (i piccolo)**: rotano molto per posizione -> sensibili a piccole distanze
- **Dimensioni alte (i grande)**: rotano poco per posizione -> sensibili a grandi distanze

Questo crea una rappresentazione multi-scala della posizione.

### Rope Base e Lunghezza del Contesto

Aumentare `base` riduce tutte le frequenze, permettendo al modello di
distinguere posizioni piu' distanti senza perdita di informazione:

| Modello | rope_base | Context Length |
|---------|-----------|----------------|
| LLaMA 1 | 10,000 | 2,048 |
| LLaMA 2 | 10,000 | 4,096 |
| LLaMA 3 | 500,000 | 128,000 |
| gpt_model4.py | 10,000 | 1,024 |
| Mistral | 10,000 | 32,768 |


## 4.4 `precompute_rope_freqs()` — Riga per Riga

```python
def precompute_rope_freqs(head_dim, context_length, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length, dtype=torch.float32)
    freqs = torch.outer(positions, theta)
    cos = torch.cos(freqs)
    sin = torch.sin(freqs)
    return cos, sin
```

**Linea 1: `theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))`**

- `torch.arange(0, head_dim, 2)`: `[0, 2, 4, ..., head_dim-2]`, shape `(head_dim//2,)` = `(32,)` per head_dim=64
- `.float() / head_dim`: normalizza a `[0, 2/64, 4/64, ..., 62/64]`
- `base ** (...)`: es. `10000^0 = 1`, `10000^(2/64) = 1.514`, `10000^(62/64) = 6607`
- `1.0 / (...)`: inverte: `theta = [1.0, 0.660, ..., 0.000151]`
- Shape: `(head_dim//2,)` = `(32,)`

**Linea 2: `positions = torch.arange(context_length, dtype=torch.float32)`**

- Crea il vettore `[0, 1, 2, ..., context_length-1]`
- Shape: `(context_length,)` = `(1024,)`

**Linea 3: `freqs = torch.outer(positions, theta)`**

- Prodotto esterno: `freqs[t, i] = positions[t] * theta[i] = t * theta_i`
- Questa e' la **tabella degli angoli di rotazione**
- `freqs[0, :] = 0` (posizione 0: nessuna rotazione)
- `freqs[1, :] = theta` (posizione 1: rotazione di theta)
- `freqs[t, :] = t * theta` (posizione t: rotazione di t*theta)
- Shape: `(context_length, head_dim//2)` = `(1024, 32)`

**Linee 4-5: `cos = torch.cos(freqs)`, `sin = torch.sin(freqs)`**

- Calcola coseno e seno di tutti gli angoli
- Valori in [-1, 1], shape invariata: `(1024, 32)` ciascuno
- Questi sono i **buffer pre-calcolati** che vengono registrati con `register_buffer`
- Pre-calcolare evita di ricalcolare cos/sin ad ogni forward pass


In [ ]:
# Capitolo 4: Demo precompute_rope_freqs
import torch

def precompute_rope_freqs(head_dim, context_length, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length, dtype=torch.float32)
    freqs = torch.outer(positions, theta)
    cos = torch.cos(freqs)
    sin = torch.sin(freqs)
    return cos, sin

# Demo con head_dim=8 (piccolo per leggibilita')
head_dim = 8
T = 6
base = 10000

# Calcolo theta
theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
print(f'head_dim={head_dim}, D/2={head_dim//2} coppie di dimensioni')
print(f'theta (frequenze) = {theta.numpy().round(4)}')
print(f'  theta[0]={theta[0]:.4f} (alta freq: ruota molto per posizione)')
print(f'  theta[-1]={theta[-1]:.4f} (bassa freq: ruota poco per posizione)')

# Prodotto esterno
positions = torch.arange(T, dtype=torch.float32)
freqs = torch.outer(positions, theta)
print(f'\nfreqs (angoli di rotazione) shape={freqs.shape}:')
print('  Righe=posizioni, Colonne=coppie di dim')
for t in range(T):
    print(f'  pos={t}: {freqs[t].numpy().round(3)}')

cos, sin = precompute_rope_freqs(head_dim, T)
print(f'\ncos shape: {cos.shape}, sin shape: {sin.shape}')
print('cos[0] (posizione 0):', cos[0].numpy().round(3), '(tutti 1: nessuna rotazione)')
print('cos[1] (posizione 1):', cos[1].numpy().round(3))
print('sin[0] (posizione 0):', sin[0].numpy().round(3), '(tutti 0: nessuna rotazione)')
print('sin[1] (posizione 1):', sin[1].numpy().round(3))


## 4.5 `apply_rope()` — Riga per Riga

```python
def apply_rope(x, cos, sin):
    _B, _H, T, D = x.shape
    half = D // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    rotated = torch.cat([
        x1 * c - x2 * s,
        x2 * c + x1 * s,
    ], dim=-1)
    return rotated
```

**Input:** `x` shape `(B, H, T, D)` — query o key

**Linea 1: `_B, _H, T, D = x.shape`**
Legge le dimensioni. T puo' essere <= context_length (per sequenze corte).
D = head_dim, es. 64.

**Linea 2: `half = D // 2`**
Divide il vettore in due meta' uguali: es. 64 // 2 = 32.

**Linee 3-4: `x1 = x[..., :half]`, `x2 = x[..., half:]`**
- `x1`: prima meta' delle dimensioni, shape `(B, H, T, D//2)`
- `x2`: seconda meta' delle dimensioni, shape `(B, H, T, D//2)`
- `...` e' Ellipsis: seleziona tutte le dimensioni precedenti

**Linee 5-6: `c = cos[:T].unsqueeze(0).unsqueeze(0)`**
- `cos[:T]`: seleziona le prime T posizioni, shape `(T, D//2)`
- `.unsqueeze(0)`: aggiunge dimensione batch, shape `(1, T, D//2)`
- `.unsqueeze(0)`: aggiunge dimensione heads, shape `(1, 1, T, D//2)`
- Broadcasting: si espande automaticamente su B e H

**Linee 7-10: La Rotazione**
```
new_x1 = x1*cos - x2*sin   # parte reale della rotazione 2D
new_x2 = x2*cos + x1*sin   # parte immaginaria della rotazione 2D
```

Questa e' esattamente la formula della rotazione 2D applicata a ogni coppia (x1[i], x2[i])!

**Linea 11: `torch.cat([...], dim=-1)`**
Ricongiungi le due meta' lungo l'ultima dimensione -> shape `(B, H, T, D)` = originale.


In [ ]:
# Capitolo 4: Demo apply_rope con valori numerici
import torch

def precompute_rope_freqs(head_dim, context_length, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length, dtype=torch.float32)
    freqs = torch.outer(positions, theta)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    _B, _H, T, D = x.shape
    half = D // 2
    x1 = x[..., :half]
    x2 = x[..., half:]
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)

torch.manual_seed(0)
B, H, T, D = 1, 1, 4, 4  # piccolo per leggibilita'
x = torch.randn(B, H, T, D)
cos, sin = precompute_rope_freqs(D, T)

print(f'Input x shape: {x.shape}')
print(f'cos shape: {cos.shape}, sin shape: {sin.shape}')

for t in range(T):
    vec = x[0, 0, t]
    print(f'  pos {t}: {vec.numpy().round(3)}')

x_rot = apply_rope(x, cos, sin)
print(f'\nOutput (dopo RoPE) shape: {x_rot.shape}')
for t in range(T):
    vec = x_rot[0, 0, t]
    print(f'  pos {t}: {vec.numpy().round(3)}')

print('\nVerifica: pos 0 dovrebbe essere identica (theta*0=0, cos=1, sin=0):')
print(f'  originale: {x[0,0,0].numpy().round(3)}')
print(f'  ruotata:   {x_rot[0,0,0].numpy().round(3)}')

# Verifica che la norma sia preservata
norms_orig = x.norm(dim=-1)
norms_rot  = x_rot.norm(dim=-1)
print(f'\nNorma preservata? (rotazione e isometrica):')
print(f'  Norme originali: {norms_orig[0,0].numpy().round(4)}')
print(f'  Norme ruotate:   {norms_rot[0,0].numpy().round(4)}')
print(f'  Max diff: {(norms_orig - norms_rot).abs().max():.6f}')


In [ ]:
# Capitolo 4: Verifica della proprieta' relativa di RoPE
# Il dot product tra query@pos_p e key@pos_q dipende solo da (p-q)
import torch

def precompute_rope_freqs(head_dim, context_length, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length, dtype=torch.float32)
    freqs = torch.outer(positions, theta)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    _B, _H, T, D = x.shape
    half = D // 2
    x1, x2 = x[..., :half], x[..., half:]
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)

torch.manual_seed(42)
D = 8
T = 10
cos_buf, sin_buf = precompute_rope_freqs(D, T)

# Vettori query e key fissi
q = torch.randn(1, 1, 1, D)  # un solo vettore query
k = torch.randn(1, 1, 1, D)  # un solo vettore key

print('Verifica proprieta RoPE:')
print('dot(RoPE(q, pos_p), RoPE(k, pos_q)) dovrebbe dipendere solo da (p-q)')
print()

# Testa varie coppie con la stessa distanza relativa = 3
distanza = 3
for p in [3, 4, 5, 6]:
    qq = q.expand(1, 1, p+1, D).clone()
    kk = k.expand(1, 1, p+1, D).clone()
    # Prendi solo la posizione p per q e (p-distanza) per k
    q_rot = apply_rope(qq, cos_buf, sin_buf)[0, 0, p, :]  # pos p
    k_rot = apply_rope(kk, cos_buf, sin_buf)[0, 0, p-distanza, :]  # pos p-distanza
    dot = (q_rot * k_rot).sum().item()
    print(f'  q@pos={p}, k@pos={p-distanza}, dist={distanza}: dot = {dot:.4f}')

print('I valori di dot product sono simili (con RoPE)')
print('(piccole differenze per precisione numerica floating point)')


## 4.6 `register_buffer`: Perche' Non e' un Parametro

In `GroupedQueryAttention.__init__`:
```python
cos, sin = precompute_rope_freqs(self.head_dim, context_length, rope_base)
self.register_buffer('rope_cos', cos)
self.register_buffer('rope_sin', sin)
```

`register_buffer` registra un tensore come **buffer** del modulo, non come parametro.

| Aspetto | `nn.Parameter` | `register_buffer` |
|---------|----------------|-------------------|
| Appreso (gradiente) | Si' | **No** |
| Incluso in `parameters()` | Si' | **No** |
| Salvato in `state_dict()` | Si' | Si' |
| Spostato con `model.to(device)` | Si' | Si' |
| Incluso in `model.buffers()` | No | Si' |

Le frequenze cos/sin sono **costanti** calcolate dalla geometria della rotazione, non da parametri appresi. Non devono cambiare durante il training. Ma devono essere salvate con il modello (state_dict) e spostate sul device giusto.

## 4.7 Confronto con Positional Embeddings Assoluti

| Aspetto | Absolute Learned (GPT-2) | RoPE |
|---------|--------------------------|------|
| Parametri extra | T*C = ~786K | 0 (buffer, non parametri) |
| Generalizzazione OOD | Pessima | Buona |
| Informazione relativa | Implicita | Esplicita |
| Dove viene applicata | Input (aggiunta a tok_emb) | Dentro attention (Q, K) |
| Estendibilita' contesto | Richiede fine-tuning | Solo cambio di rope_base |
| Max seq in training | Fisso | Interpolabile |


---
# Capitolo 5: GroupedQueryAttention (GQA)

## 5.1 Self-Attention: Derivazione Matematica Completa

L'attenzione e' il meccanismo che permette ad ogni posizione di 'guardare' tutte le altre posizioni della sequenza.

### Proiezioni Lineari Q, K, V

Dato l'input X di shape (T, C), si calcolano tre proiezioni:
```
Q = X @ W_Q    shape: (T, d_k)
K = X @ W_K    shape: (T, d_k)
V = X @ W_V    shape: (T, d_v)
```

**Intuizione:**
- Q (Query): 'Cosa sto cercando?'
- K (Key): 'Cosa offro?'
- V (Value): 'Cosa fornisco se selezionato?'

### Scaled Dot-Product Attention

```
Attention(Q, K, V) = softmax(Q @ K.T / sqrt(d_k)) @ V
```

**Perche' dividere per sqrt(d_k)?**

Se Q e K hanno componenti con varianza 1, i prodotti interni Q @ K.T hanno varianza d_k (somma di d_k prodotti di varianza 1). Per grandi d_k, questi valori diventano molto grandi, facendo saturare il softmax nelle zone dove il gradiente e' quasi zero (problema del vanishing gradient).

Dividendo per sqrt(d_k), riportiamo la varianza a 1:
```
Var(q . k / sqrt(d_k)) = Var(q . k) / d_k = d_k / d_k = 1
```

### Causal Masking (per generazione autoregressiva)

Il token in posizione t non deve 'vedere' le posizioni t+1, t+2, etc. (i token futuri non sono ancora stati generati).

Si aggiunge una maschera triangolare superiore con -inf:
```
attn_scores[i,j] = -inf   se j > i  (token futuro)
```

Dopo il softmax, exp(-inf) = 0, quindi queste posizioni hanno peso 0.

### Output

```
output = softmax_weights @ V
```

Ogni riga dell'output e' una **media pesata** dei valori V, con i pesi dati dalla compatibilita' tra Q e K.


## 5.2 Multi-Head Attention (MHA): Teoria

L'attenzione a testa singola calcola un solo pattern di attenzione. **Multi-Head Attention** esegue H attenzioni in parallelo, ciascuna su una sotto-spazio di dimensione D = C/H.

```
head_i = Attention(Q_i, K_i, V_i)
MHA(X) = concat(head_1, ..., head_H) @ W_O
```

**Vantaggi:**
- Ogni testa puo' specializzarsi su diversi tipi di relazioni
  (es. sintassi, coreference, prossimita' posizionale)
- Arricchisce la rappresentazione senza aumentare il costo computazionale totale

**Parametri MHA (C=768, H=12, D=64):**
```
W_Q: (C, H*D) = (768, 768)  -> 589,824 params
W_K: (C, H*D) = (768, 768)  -> 589,824 params
W_V: (C, H*D) = (768, 768)  -> 589,824 params
W_O: (H*D, C) = (768, 768)  -> 589,824 params
Totale:                      -> 2,359,296 params per layer
```

## 5.3 Il Problema della KV-Cache nell'Inferenza Autoregressiva

Durante la **generazione di testo**, il modello predice un token alla volta:
```
Step 1: input=[token_0]          -> predice token_1
Step 2: input=[token_0, token_1] -> predice token_2
Step 3: input=[token_0, token_1, token_2] -> predice token_3
...
```

Ad ogni step, ricalcoliamo K e V per **tutti** i token precedenti. Ma K[t] e V[t] non cambiano tra gli step! Possiamo **cachare** K e V:

```
KV-cache size per layer = T * H * D * 2 (K e V)
                        = T * 12 * 64 * 2 (per GPT-2 Small)
                        = 1536 * T floats per layer
Per 12 layer e T=1024:  = 12 * 1536 * 1024 = 18,874,368 floats
In float16:             = 37.7 MB
```

Con modelli piu' grandi (es. LLaMA 70B, H=64, D=128, L=80):
```
KV-cache per T=4096: 80 * 4096 * 64 * 128 * 2 * 2 = 10.7 GB
```

La KV-cache e' il **collo di bottiglia** dell'inferenza per batch grandi!


## 5.4 Multi-Query Attention e Grouped Query Attention

### Multi-Query Attention (MQA, Shazeer 2019)

Riduce i KV heads a **1 solo**: tutti i Q heads condividono lo stesso K e V.

```
Parametri K: (C, 1*D)  invece di (C, H*D)
Parametri V: (C, 1*D)  invece di (C, H*D)
KV-cache ridotta di H volte!
```

**Costo:** leggera perdita di qualita' (1-3 punti perplexity).

### Grouped Query Attention (GQA, Ainslie et al. 2023)

Compromesso tra MHA e MQA: usa G < H KV heads.

```
H = n_heads = 12 (query heads)
G = n_kv_heads = 4 (key/value heads)
n_rep = H / G = 3 (ogni KV head condiviso da 3 Q heads)

Gruppo 1: Q heads 0,1,2  condividono K_0, V_0
Gruppo 2: Q heads 3,4,5  condividono K_1, V_1
Gruppo 3: Q heads 6,7,8  condividono K_2, V_2
Gruppo 4: Q heads 9,10,11 condividono K_3, V_3
```

**KV-cache riduzione:** H/G = 12/4 = **3 volte** meno memoria!

**Qualita':** quasi identica a MHA (0.5-1 punto perplexity di differenza)

### Tabella Comparativa

| Metodo | KV Heads | KV-cache | Qualita' | Usato in |
|--------|----------|----------|----------|----------|
| MHA | H=12 | 1x | Massima | GPT-2, LLaMA 1 |
| GQA | G=4 | G/H = 1/3x | ~MHA | LLaMA 2 70B, LLaMA 3 8B, Mistral |
| MQA | 1 | 1/H = 1/12x | -2-3pp | PaLM |

### `repeat_interleave`: Come Espandere i KV Heads

GQA ha G=4 KV heads ma H=12 Q heads. Per il calcolo dell'attenzione, dobbiamo espandere K e V da G a H:

```python
keys = keys.repeat_interleave(n_rep, dim=1)  # (B,G,T,D) -> (B,H,T,D)
```

`repeat_interleave` con `n_rep=3` su dim=1 trasforma:
```
[head_0, head_1, head_2, head_3]  (G=4 elementi)
->
[head_0, head_0, head_0, head_1, head_1, head_1, ...head_3]  (H=12 elementi)
```

Ogni KV head e' ripetuto 3 volte, condiviso tra 3 Q heads.


## 5.5 `GroupedQueryAttention.__init__` — Riga per Riga

```python
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,
                 n_heads, n_kv_heads, rope_base=10000, qkv_bias=False):
        super().__init__()
        assert d_out % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_out = d_out          # C=768
        self.n_heads = n_heads      # H=12
        self.n_kv_heads = n_kv_heads  # G=4
        self.n_rep = n_heads // n_kv_heads   # 3
        self.head_dim = d_out // n_heads     # D=64
```

**Assertions:**
- `d_out % n_heads == 0`: head_dim deve essere intero (768/12 = 64 OK)
- `n_heads % n_kv_heads == 0`: n_rep deve essere intero (12/4 = 3 OK)

**Attributi derivati:**
- `n_rep = 3`: ogni KV head serve 3 Q heads
- `head_dim = 64`: dimensione di ogni testa

```python
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)          # (768,768)
        self.W_key   = nn.Linear(d_in, n_kv_heads*self.head_dim, bias=qkv_bias)  # (768,256)
        self.W_value = nn.Linear(d_in, n_kv_heads*self.head_dim, bias=qkv_bias)  # (768,256)
        self.out_proj = nn.Linear(d_out, d_out, bias=False)            # (768,768)
```

**Dimensioni delle proiezioni:**
- `W_query`: C -> H*D = 768 -> 768 (H=12 query heads)
- `W_key`: C -> G*D = 768 -> 256 (G=4 KV heads, 3x meno parametri)
- `W_value`: C -> G*D = 768 -> 256 (stessa riduzione)

**Risparmio parametri rispetto a MHA:**
- MHA: W_K + W_V = 2 * 768*768 = 1,179,648 params
- GQA: W_K + W_V = 2 * 768*256 = 393,216 params (3x meno!)

```python
        self.register_buffer('mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
        cos, sin = precompute_rope_freqs(self.head_dim, context_length, rope_base)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)
```

- `mask`: buffer (T,T), 1=futuro da mascherare, 0=visibile
- `rope_cos`, `rope_sin`: buffer (T, D//2) per RoPE
- Tutti e tre sono buffer (non parametri): vengono spostati con il modello ma non appresi


## 5.6 `GroupedQueryAttention.forward` — Tutti i 6 Passi

```python
    def forward(self, x):
        B, T, _C = x.shape
```

**Input:** x shape `(B, T, C)` es. `(2, 256, 768)`

### Passo 1: Proiezioni Q, K, V

```python
        queries = self.W_query(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        # (B,T,C) -> (B,T,H*D) -> (B,T,H,D) -> (B,H,T,D)
        keys    = self.W_key(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        # (B,T,C) -> (B,T,G*D) -> (B,T,G,D) -> (B,G,T,D)
        values  = self.W_value(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        # (B,T,C) -> (B,T,G*D) -> (B,T,G,D) -> (B,G,T,D)
```

Il `.view(B, T, H, D)` separa la dimensione H*D nelle H teste.
Il `.transpose(1, 2)` porta le teste prima di T per parallelismo.

### Passo 2: Applica RoPE

```python
        queries = apply_rope(queries, self.rope_cos, self.rope_sin)  # (B,H,T,D)
        keys    = apply_rope(keys,    self.rope_cos, self.rope_sin)  # (B,G,T,D)
```

RoPE viene applicata **solo a Q e K**, non a V. I valori V non vengono ruotati perche' la posizione deve influenzare il punteggio di attenzione (Q.K), non il contenuto dei valori.

### Passo 3: Espandi KV da G a H Teste

```python
        keys   = keys.repeat_interleave(self.n_rep, dim=1)   # (B,G,T,D) -> (B,H,T,D)
        values = values.repeat_interleave(self.n_rep, dim=1) # (B,G,T,D) -> (B,H,T,D)
```

Questo e' il cuore di GQA: i 4 KV heads vengono 'clonati' 3 volte ciascuno
per ottenere 12 coppie (K, V), una per ogni Q head.

### Passo 4: Scaled Dot-Product Attention con Maschera Causale

```python
        attn_scores = queries @ keys.transpose(2, 3)  # (B,H,T,D)@(B,H,D,T) -> (B,H,T,T)
        mask_bool = self.mask.bool()[:T, :T]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)  # (B,H,T,T)
        attn_weights = self.dropout(attn_weights)
```

- `queries @ keys.transpose(2,3)`: prodotto scalare tra ogni Q e ogni K
- `masked_fill_(-inf)`: maschera i token futuri (autoregressivo)
- `softmax / sqrt(head_dim)`: normalizzazione scalata

### Passo 5: Aggregazione e Proiezione Output

```python
        context_vec = (attn_weights @ values).transpose(1, 2)
        # (B,H,T,T) @ (B,H,T,D) -> (B,H,T,D) -> (B,T,H,D)
        context_vec = context_vec.reshape(B, T, self.d_out)
        # (B,T,H,D) -> (B,T,H*D) = (B,T,C)
        context_vec = self.out_proj(context_vec)
        # (B,T,C) -> (B,T,C)
        return context_vec
```


In [ ]:
# Capitolo 5: Demo GQA - forward pass completo
import torch
import torch.nn as nn
import torch.nn.functional as F

def precompute_rope_freqs(head_dim, context_length, base=10000):
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    positions = torch.arange(context_length, dtype=torch.float32)
    freqs = torch.outer(positions, theta)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    _B, _H, T, D = x.shape
    half = D // 2
    x1, x2 = x[..., :half], x[..., half:]
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c - x2*s, x2*c + x1*s], dim=-1)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout,
                 n_heads, n_kv_heads, rope_base=10000, qkv_bias=False):
        super().__init__()
        assert d_out % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_out = d_out
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d_out // n_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, n_kv_heads*self.head_dim, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, n_kv_heads*self.head_dim, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1))
        cos, sin = precompute_rope_freqs(self.head_dim, context_length, rope_base)
        self.register_buffer('rope_cos', cos)
        self.register_buffer('rope_sin', sin)

    def forward(self, x):
        B, T, _C = x.shape
        queries = self.W_query(x).view(B, T, self.n_heads, self.head_dim).transpose(1,2)
        keys    = self.W_key(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1,2)
        values  = self.W_value(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1,2)
        queries = apply_rope(queries, self.rope_cos, self.rope_sin)
        keys    = apply_rope(keys, self.rope_cos, self.rope_sin)
        keys   = keys.repeat_interleave(self.n_rep, dim=1)
        values = values.repeat_interleave(self.n_rep, dim=1)
        attn_scores = queries @ keys.transpose(2,3)
        mask_bool = self.mask.bool()[:T,:T]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1,2)
        context_vec = context_vec.reshape(B, T, self.d_out)
        return self.out_proj(context_vec)

torch.manual_seed(42)
B, T, C = 2, 8, 64  # piccolo per leggibilita'
cfg = dict(d_in=C, d_out=C, context_length=T, dropout=0.0,
           n_heads=4, n_kv_heads=2, rope_base=10000, qkv_bias=False)
gqa = GroupedQueryAttention(**cfg)
gqa.eval()

x = torch.randn(B, T, C)
with torch.no_grad():
    out = gqa(x)

print(f'Input shape:  {x.shape}')
print(f'Output shape: {out.shape}')
print(f'n_heads={cfg["n_heads"]}, n_kv_heads={cfg["n_kv_heads"]}, n_rep={gqa.n_rep}')
print(f'head_dim={gqa.head_dim}')
print()
print('Parametri per componente:')
print(f'  W_query: {gqa.W_query.weight.numel():,} ({list(gqa.W_query.weight.shape)})')
print(f'  W_key:   {gqa.W_key.weight.numel():,} ({list(gqa.W_key.weight.shape)})')
print(f'  W_value: {gqa.W_value.weight.numel():,} ({list(gqa.W_value.weight.shape)})')
print(f'  out_proj:{gqa.out_proj.weight.numel():,} ({list(gqa.out_proj.weight.shape)})')
total = sum(p.numel() for p in gqa.parameters())
print(f'  Totale:  {total:,}')
print()
print('Buffer registrati:')
for name, buf in gqa.named_buffers():
    print(f'  {name}: shape={list(buf.shape)}')


In [ ]:
# Capitolo 5: Confronto MHA vs GQA — parametri e KV-cache
C, H, G, T, L = 768, 12, 4, 1024, 12
D = C // H  # head_dim = 64

print('Confronto MHA vs GQA (C=768, H=12, G=4, T=1024, L=12 layers)')
print('=' * 60)

# Parametri
mha_wq = C * H * D; mha_wk = C * H * D; mha_wv = C * H * D; mha_wo = H*D * C
gqa_wq = C * H * D; gqa_wk = C * G * D; gqa_wv = C * G * D; gqa_wo = H*D * C

print('Parametri per layer di attenzione:')
print(f'  MHA: W_Q={mha_wq:,} + W_K={mha_wk:,} + W_V={mha_wv:,} + W_O={mha_wo:,}')
print(f'       Totale = {mha_wq+mha_wk+mha_wv+mha_wo:,}')
print(f'  GQA: W_Q={gqa_wq:,} + W_K={gqa_wk:,} + W_V={gqa_wv:,} + W_O={gqa_wo:,}')
print(f'       Totale = {gqa_wq+gqa_wk+gqa_wv+gqa_wo:,}')
ratio_params = (gqa_wq+gqa_wk+gqa_wv+gqa_wo)/(mha_wq+mha_wk+mha_wv+mha_wo)
print(f'  GQA/MHA params = {ratio_params:.3f} ({(1-ratio_params)*100:.1f}% meno parametri)')

# KV-cache
mha_kvcache = T * H * D * 2 * L  # K e V per ogni layer
gqa_kvcache = T * G * D * 2 * L
print()
print('KV-cache durante inferenza (float16, 2 bytes):')
print(f'  MHA: {mha_kvcache:,} floats = {mha_kvcache*2/1e6:.2f} MB')
print(f'  GQA: {gqa_kvcache:,} floats = {gqa_kvcache*2/1e6:.2f} MB')
print(f'  Riduzione: {H/G:.0f}x meno memoria KV-cache')

# Per LLaMA 70B
print()
print('Per LLaMA 70B (H=64, G=8, D=128, L=80, T=4096):')
H2, G2, D2, L2, T2 = 64, 8, 128, 80, 4096
mha2 = T2*H2*D2*2*L2*2/1e9
gqa2 = T2*G2*D2*2*L2*2/1e9
print(f'  MHA KV-cache: {mha2:.1f} GB')
print(f'  GQA KV-cache: {gqa2:.2f} GB (riduzione {H2/G2:.0f}x)')


---
# Capitolo 6: FeedForwardSwiGLU

## 6.1 Il Ruolo delle FFN nel Transformer

Ogni blocco Transformer ha due componenti:
1. **Self-Attention**: mescola informazioni tra posizioni diverse della sequenza
2. **Feed-Forward Network (FFN)**: trasforma ogni posizione **indipendentemente**

L'FFN processa ogni token separatamente (stesso MLP applicato a ogni posizione). Recenti ricerche (Geva et al., 2021) suggeriscono che le FFN fungono da **memoria delle conoscenze fattuali**: 'Roma e' la capitale dell'Italia', 'il cielo e' blu', etc. vengono codificate principalmente nei pesi dell'FFN.

### FFN in GPT-2 (GELU)

```python
# GPT-2 usa 2 proiezioni lineari con GELU:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        self.layers = nn.Sequential(
            nn.Linear(C, 4*C),  # espansione 4x
            nn.GELU(),
            nn.Linear(4*C, C),  # proiezione di ritorno
        )
```

**GELU (Gaussian Error Linear Unit):**
```
GELU(x) = x * Phi(x)
```
dove Phi(x) e' la CDF della distribuzione normale standard.
Approssimazione: `GELU(x) ≈ 0.5 * x * (1 + tanh(sqrt(2/pi) * (x + 0.044715 * x^3)))`

GELU e' una funzione smooth che 'stochasticamente' azzera i valori negativi,
simile a ReLU ma differenziabile ovunque.


## 6.2 Gated Linear Units e SwiGLU

### Gated Linear Units (Dauphin et al., 2017)

L'idea dei **gate** nelle reti neurali: controllare quale informazione passa.

```
GLU(x) = (W1 @ x + b1) * sigmoid(W2 @ x + b2)
```

Il primo termine e' il 'contenuto', il secondo e' il 'gate' (0-1).
Il gate controlla quanta informazione passa: se gate ≈ 0, blocca il segnale;
se gate ≈ 1, lascia passare tutto.

### SwiGLU (Shazeer, 2020)

'GLU Variants Improve Transformer' testa diverse funzioni di attivazione nel gate:

```
SwiGLU(x) = SiLU(W_gate @ x) * (W_up @ x)
```

Dove:
- `SiLU(x) = x * sigmoid(x)`: la funzione Swish (auto-gated)
- `W_gate @ x`: il segnale di gating
- `W_up @ x`: il segnale di contenuto
- Prodotto elemento per elemento: gate * content

Poi proietta di nuovo:
```
output = W_down @ SwiGLU(x)
```

### Tre Matrici invece di Due

Rispetto al FFN standard (2 matrici: W1 e W2), SwiGLU usa **3 matrici**:
W_gate, W_up, W_down. Per mantenere lo stesso numero di parametri totali:

```
FFN standard:    2 * C * (4C) = 8C^2 parametri
SwiGLU (3 matr): 3 * C * H    = 3CH parametri
Per 3CH = 8C^2: H = 8C/3 ≈ 2.667C
```

Quindi hidden_dim = 8/3 * C per lo stesso budget di parametri!


## 6.3 Il Calcolo di `hidden_dim`

```python
hidden_dim = int(math.ceil((8 / 3 * emb_dim) / 256) * 256)
```

**Perche' arrotondare al multiplo di 256?**

Le GPU moderne eseguono le operazioni matriciali in modo ottimale quando le dimensioni sono multipli di certi valori (tipicamente 64, 128, 256):
- I **Tensor Cores** di NVIDIA lavorano con tile di 16x16 o 32x32
- Dimensioni non allineate causano 'padding' invisibile che spreca cicli di calcolo
- Multipli di 256 garantiscono allineamento per tutte le GPU NVIDIA moderne

**Esempi:**

| emb_dim | 8/3 * C (raw) | /256 | ceil | *256 = hidden_dim |
|---------|---------------|------|------|--------------------|
| 256 | 682.67 | 2.667 | 3 | **768** |
| 384 | 1024.0 | 4.0 | 4 | **1024** |
| 512 | 1365.33 | 5.333 | 6 | **1536** |
| 768 | 2048.0 | 8.0 | 8 | **2048** |
| 1024 | 2730.67 | 10.667 | 11 | **2816** |

Per C=768: `8/3 * 768 = 2048.0`, gia' multiplo di 256, quindi hidden_dim = 2048.

## 6.4 Implementazione: `class FeedForwardSwiGLU` — Riga per Riga

```python
class FeedForwardSwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        emb_dim = cfg['emb_dim']
        hidden_dim = int(math.ceil((8 / 3 * emb_dim) / 256) * 256)
        self.gate = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.up   = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.down = nn.Linear(hidden_dim, emb_dim, bias=False)
```

- `self.gate`: W_gate, proiezione C -> H_ff per il segnale di gating
- `self.up`: W_up, proiezione C -> H_ff per il segnale di contenuto
- `self.down`: W_down, proiezione H_ff -> C di ritorno
- Tutte con `bias=False`: convenzione LLaMA, riduce parametri

```python
    def forward(self, x):
        gated = F.silu(self.gate(x)) * self.up(x)
        return self.down(gated)
```

**`gated = F.silu(self.gate(x)) * self.up(x)`**
- `self.gate(x)`: proiezione lineare, shape `(B, T, C)` -> `(B, T, H_ff)`
- `F.silu(...)`: SiLU element-wise, shape invariata
- `self.up(x)`: seconda proiezione, shape `(B, T, H_ff)`
- `* `: prodotto elemento per elemento (gating), shape `(B, T, H_ff)`

**`return self.down(gated)`**
- Proiezione di ritorno: `(B, T, H_ff)` -> `(B, T, C)`


In [ ]:
# Capitolo 6: Demo FeedForwardSwiGLU
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeedForwardSwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        emb_dim = cfg['emb_dim']
        hidden_dim = int(math.ceil((8/3 * emb_dim)/256)*256)
        self.hidden_dim = hidden_dim
        self.gate = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.up   = nn.Linear(emb_dim, hidden_dim, bias=False)
        self.down = nn.Linear(hidden_dim, emb_dim, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

cfg = {'emb_dim': 768}
ffn = FeedForwardSwiGLU(cfg)

print(f'emb_dim = {cfg["emb_dim"]}')
print(f'hidden_dim = {ffn.hidden_dim}')
print(f'  8/3 * 768 = {8/3*768:.2f} (gia multiplo di 256)')

B, T, C = 2, 16, 768
x = torch.randn(B, T, C)
with torch.no_grad():
    out = ffn(x)
print(f'\nInput shape: {x.shape}')
print(f'Output shape: {out.shape}')

print('\nParametri per matrice:')
print(f'  gate: {ffn.gate.weight.numel():,}  {list(ffn.gate.weight.shape)}')
print(f'  up:   {ffn.up.weight.numel():,}  {list(ffn.up.weight.shape)}')
print(f'  down: {ffn.down.weight.numel():,}  {list(ffn.down.weight.shape)}')
total = sum(p.numel() for p in ffn.parameters())
print(f'  Totale SwiGLU: {total:,}')

# Confronto con FFN standard (GELU, 4x)
class FFN_GELU(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.w1 = nn.Linear(C, 4*C)
        self.w2 = nn.Linear(4*C, C)
    def forward(self, x):
        return self.w2(F.gelu(self.w1(x)))

ffn_gelu = FFN_GELU(768)
total_gelu = sum(p.numel() for p in ffn_gelu.parameters())
print(f'\n  Totale GELU (4x): {total_gelu:,}')
print(f'  Differenza: {total - total_gelu:,} ({(total-total_gelu)/total_gelu*100:.1f}%)')
print(f'\nConclusion: parametri quasi identici (SwiGLU ha hidden_dim=2048 vs 4*768=3072)')
print('ma qualita empiricamente superiore')


In [ ]:
# Capitolo 6: Visualizzazione del meccanismo SwiGLU
import torch
import torch.nn.functional as F

# Mostra come il gate controlla il flusso
x_demo = torch.linspace(-3, 3, 100)

# Simulazione del gate: cosa succede con gate positivo vs negativo
gate_pos = torch.ones_like(x_demo) * 2.0   # gate alto
gate_neg = torch.ones_like(x_demo) * (-2.0) # gate basso
up_signal = x_demo  # segnale di contenuto

gated_pos = F.silu(gate_pos) * up_signal
gated_neg = F.silu(gate_neg) * up_signal

print('Effetto del gate su un segnale lineare (up = x):')
for i in [25, 50, 75]:
    xv = x_demo[i].item()
    print(f'  x={xv:.2f}: gate_pos*content={gated_pos[i]:.3f}, gate_neg*content={gated_neg[i]:.3f}')

print('\nSiLU: x * sigmoid(x)')
x_test = torch.tensor([-2., -1., 0., 1., 2.])
print(f'  x:         {x_test.numpy()}')
print(f'  sigmoid(x):{torch.sigmoid(x_test).numpy().round(3)}')
print(f'  SiLU(x):   {F.silu(x_test).numpy().round(3)}')
print(f'  ReLU(x):   {F.relu(x_test).numpy().round(3)}')
print()
print('SiLU vs ReLU: SiLU e smooth (differenziabile in 0) e ha valori negativi lievi')
print('Questo permette gradienti non-zero anche per x < 0 (zona di saturazione)')


---
# Capitolo 7: TransformerBlock4

## 7.1 Il Blocco Transformer Completo

`TransformerBlock4` e' il blocco modulare che viene ripetuto 12 volte in `GPT4Model`. Ogni blocco contiene:
1. Pre-RMSNorm
2. GroupedQueryAttention (GQA + RoPE)
3. Residual connection
4. Pre-RMSNorm
5. FeedForwardSwiGLU
6. Residual connection

## 7.2 Pre-Norm vs Post-Norm

### Post-Norm (Vaswani 2017 originale)

```
x -> Attention(x) -> residual = x + Attention(x) -> LayerNorm(residual)
```

**Problema:** Con modelli profondi (>12 layer), il training post-norm diventa instabile:
- Le varianze delle attivazioni crescono con la profondita'
- Richiede warmup molto lungo e learning rate minuscolo
- Facilmente porta a NaN durante il training

### Pre-Norm (GPT-2, LLaMA, gpt_model4.py)

```
x -> RMSNorm(x) -> Attention -> residual = x + Attention(RMSNorm(x))
```

**Vantaggi:**
- Il segnale residuale `x` fluisce **direttamente** senza normalizzazione
- Training piu' stabile, meno sensibile all'inizializzazione
- Meno necessita' di warmup lungo
- Gradiente sempre definito attraverso il ramo residuale

## 7.3 Residual Connections

Le skip connections (He et al., ResNet 2016) sono cruciali per addestrare reti profonde:

```
output = x + sublayer(norm(x))
```

**Perche' funzionano:**
Durante il backprop, il gradiente ha un **highway diretto**:
```
∂L/∂x = ∂L/∂output * (1 + ∂sublayer/∂x)
```

Il termine `1` garantisce che il gradiente possa sempre fluire, anche se `sublayer` ha gradienti piccoli. Questo risolve il vanishing gradient nei Transformer profondi.

**Intuizione geometrica:** Il residual connette direttamente l'input all'output, come un 'elevatore' che bypassa i layer. Il sublayer impara solo la **correzione** rispetto all'identita', non la trasformazione completa. Questo e' molto piu' facile da imparare!


## 7.4 `TransformerBlock4.__init__` — Riga per Riga

```python
class TransformerBlock4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            dropout=cfg['drop_rate'],
            n_heads=cfg['n_heads'],
            n_kv_heads=cfg['n_kv_heads'],
            rope_base=cfg.get('rope_base', 10000),
            qkv_bias=cfg.get('qkv_bias', False),
        )
        self.ff    = FeedForwardSwiGLU(cfg)
        self.norm1 = RMSNorm(cfg['emb_dim'])
        self.norm2 = RMSNorm(cfg['emb_dim'])
        self.drop_shortcut = nn.Dropout(cfg['drop_rate'])
```

**`self.att`:** L'istanza GQA con tutti i parametri dal config.
Nota: `cfg.get('rope_base', 10000)` usa il default 10000 se non specificato.

**`self.ff`:** L'FFN SwiGLU.

**`self.norm1`, `self.norm2`:** Due istanze **separate** di RMSNorm.
Sono separate perche' imparano scale diverse:
- `norm1`: normalizza prima dell'attention
- `norm2`: normalizza prima del FFN

**`self.drop_shortcut`:** Un'**unica** istanza di Dropout riusata per entrambi i blocchi.
Dropout e' stateless tra chiamate (non ha memoria), quindi condividerlo e' sicuro
e riduce il numero di oggetti.


## 7.5 `TransformerBlock4.forward` — Riga per Riga

```python
    def forward(self, x):
        # Blocco Attenzione
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        # Blocco Feed-Forward
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x
```

### Blocco Attenzione (5 passi):

1. **`shortcut = x`**: Salva il residual prima di qualsiasi trasformazione, shape `(B,T,C)`
2. **`x = self.norm1(x)`**: Pre-normalizzazione RMS, shape invariata `(B,T,C)`
3. **`x = self.att(x)`**: GroupedQueryAttention con RoPE, shape invariata `(B,T,C)`
4. **`x = self.drop_shortcut(x)`**: Dropout (solo durante training), shape invariata
5. **`x = x + shortcut`**: Residual connection, shape invariata `(B,T,C)`

### Blocco Feed-Forward (5 passi):

1. **`shortcut = x`**: Salva il nuovo residual (output del blocco attenzione)
2. **`x = self.norm2(x)`**: Pre-normalizzazione RMS per il FFN
3. **`x = self.ff(x)`**: FeedForwardSwiGLU, shape invariata
4. **`x = self.drop_shortcut(x)`**: Dropout
5. **`x = x + shortcut`**: Secondo residual

**Analisi delle shape:**
```
Input:   (B, T, C) = (2, 1024, 768)
norm1:   (B, T, C) invariata
att:     (B, T, C) invariata (attenzione mescola le posizioni ma mantiene C)
dropout: (B, T, C) invariata
+shortcut: (B, T, C) invariata
norm2:   (B, T, C) invariata
ff:      (B, T, C) invariata (FFN trasforma ogni token indip., mantiene C)
dropout: (B, T, C) invariata
+shortcut: (B, T, C) invariata
Output:  (B, T, C) invariata
```

Il blocco Transformer e' una **funzione identita' arricchita**: input e output hanno la stessa shape, e il residual garantisce che la funzione possa imparare a essere quasi-identita' all'inizio del training.


In [ ]:
# Capitolo 7: Demo TransformerBlock4 completo
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# Definizioni complete (copiate per autonomia della cella)
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.scale * (x / rms)

def precompute_rope_freqs(head_dim, ctx, base=10000):
    theta = 1.0/(base**(torch.arange(0,head_dim,2).float()/head_dim))
    pos = torch.arange(ctx, dtype=torch.float32)
    freqs = torch.outer(pos, theta)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    _B,_H,T,D = x.shape; half=D//2
    x1,x2 = x[...,:half], x[...,half:]
    c=cos[:T].unsqueeze(0).unsqueeze(0); s=sin[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c-x2*s, x2*c+x1*s], dim=-1)

class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, ctx, drop, n_heads, n_kv, rope_base=10000, qkv_bias=False):
        super().__init__()
        self.n_heads=n_heads; self.n_kv=n_kv
        self.n_rep=n_heads//n_kv; self.hd=d_out//n_heads; self.d_out=d_out
        self.wq=nn.Linear(d_in,d_out,bias=qkv_bias)
        self.wk=nn.Linear(d_in,n_kv*self.hd,bias=qkv_bias)
        self.wv=nn.Linear(d_in,n_kv*self.hd,bias=qkv_bias)
        self.wo=nn.Linear(d_out,d_out,bias=False); self.drop=nn.Dropout(drop)
        self.register_buffer('mask',torch.triu(torch.ones(ctx,ctx),diagonal=1))
        cos,sin=precompute_rope_freqs(self.hd,ctx,rope_base)
        self.register_buffer('rc',cos); self.register_buffer('rs',sin)
    def forward(self, x):
        B,T,_=x.shape
        q=self.wq(x).view(B,T,self.n_heads,self.hd).transpose(1,2)
        k=self.wk(x).view(B,T,self.n_kv,self.hd).transpose(1,2)
        v=self.wv(x).view(B,T,self.n_kv,self.hd).transpose(1,2)
        q=apply_rope(q,self.rc,self.rs); k=apply_rope(k,self.rc,self.rs)
        k=k.repeat_interleave(self.n_rep,1); v=v.repeat_interleave(self.n_rep,1)
        sc=q@k.transpose(2,3); sc.masked_fill_(self.mask.bool()[:T,:T],-torch.inf)
        w=self.drop(torch.softmax(sc/self.hd**0.5,dim=-1))
        return self.wo((w@v).transpose(1,2).reshape(B,T,self.d_out))

class FeedForwardSwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        C=cfg['emb_dim']; H=int(math.ceil(8/3*C/256)*256)
        self.gate=nn.Linear(C,H,bias=False); self.up=nn.Linear(C,H,bias=False)
        self.down=nn.Linear(H,C,bias=False)
    def forward(self,x): return self.down(F.silu(self.gate(x))*self.up(x))

class TransformerBlock4(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att=GroupedQueryAttention(cfg['emb_dim'],cfg['emb_dim'],
            cfg['context_length'],cfg['drop_rate'],cfg['n_heads'],cfg['n_kv_heads'],
            cfg.get('rope_base',10000),cfg.get('qkv_bias',False))
        self.ff=FeedForwardSwiGLU(cfg)
        self.norm1=RMSNorm(cfg['emb_dim']); self.norm2=RMSNorm(cfg['emb_dim'])
        self.drop=nn.Dropout(cfg['drop_rate'])
    def forward(self,x):
        sc=x; x=self.drop(self.att(self.norm1(x)))+sc
        sc=x; x=self.drop(self.ff(self.norm2(x)))+sc
        return x

cfg = {'emb_dim':128,'context_length':32,'n_heads':4,'n_kv_heads':2,
       'drop_rate':0.0,'qkv_bias':False,'rope_base':10000}
block = TransformerBlock4(cfg)
block.eval()

x = torch.randn(2, 16, 128)
with torch.no_grad():
    out = block(x)

print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Shape invariata: {x.shape == out.shape}')

total = sum(p.numel() for p in block.parameters())
print(f'\nParametri totali del blocco: {total:,}')
print(f'  att: {sum(p.numel() for p in block.att.parameters()):,}')
print(f'  ff:  {sum(p.numel() for p in block.ff.parameters()):,}')
print(f'  norm1: {sum(p.numel() for p in block.norm1.parameters()):,}')
print(f'  norm2: {sum(p.numel() for p in block.norm2.parameters()):,}')


---
# Capitolo 8: GPT4Model

## 8.1 Architettura Complessiva

`GPT4Model` e' il modello completo: assemblaggio di tutti i componenti in una pipeline coerente. E' la classe che si istanzia per fare training o inferenza.

**Differenza chiave rispetto a GPTModel (GPT-2 style):**

| Aspetto | GPTModel (GPT-2) | GPT4Model |
|---------|------------------|----------|
| Positional embedding | `pos_emb: nn.Embedding(T, C)` | **Nessuno** |
| Come si codifica la posizione | Addizione all'input | RoPE dentro l'attention |
| Generalizzazione oltre T | Impossibile | Possibile con rope_base |

## 8.2 `GPT4Model.__init__` — Riga per Riga

```python
class GPT4Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'])
```

**`tok_emb: nn.Embedding(50257, 768)`**

- Tabella di look-up: ogni token ID (0-50256) mappa a un vettore C=768
- Shape dei pesi: `(vocab_size, emb_dim)` = `(50257, 768)` = **38.6M parametri**
- Inizializzazione: N(0, 1) (distribuzione normale standard)
- Durante il forward: `tok_emb(idx)` esegue una selezione di righe
- Il gradiente si propaga **solo alle righe usate** in quel batch (sparse gradient)

```python
        self.drop_emb = nn.Dropout(cfg['drop_rate'])
```

Dropout applicato agli embedding: aiuta la regolarizzazione impedendo al modello di dipendere troppo da singole componenti dell'embedding.

```python
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock4(cfg) for _ in range(cfg['n_layers'])]
        )
```

- List comprehension: crea 12 istanze distinte di `TransformerBlock4`
- `*[...]`: unpacking della lista come argomenti positional
- `nn.Sequential`: applica i blocchi in sequenza nel forward
- Ogni blocco ha i **propri parametri**: non c'e' weight tying tra blocchi

```python
        self.final_norm = RMSNorm(cfg['emb_dim'])
        self.out_head = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False)
        self.context_length = cfg['context_length']
```

**`final_norm`:** RMSNorm applicata dopo tutti i blocchi Transformer, prima della testa.
Cruciale per la stabilita' dei logit: senza normalizzazione finale, i logit potrebbero avere scala arbitraria.

**`out_head`:** Proiezione finale `(C=768, vocab_size=50257)`. Mappa ogni vettore di contesto nello spazio del vocabolario.
- `bias=False`: convenzione LLaMA
- Parametri: `768 * 50257 = 38.6M`

**`context_length`:** Salvato come attributo per accesso esterno (usato dal loop di generazione per sapere la lunghezza massima).


## 8.3 `GPT4Model.forward` — Riga per Riga

```python
    def forward(self, in_idx):
        tok_embeds = self.tok_emb(in_idx)
        x = self.drop_emb(tok_embeds)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits
```

**Input: `in_idx` shape `(B, T)`**
Indici token interi, es. `[[15496, 616, 3, 0], [464, 920, 318, 1234]]`

**`tok_embeds = self.tok_emb(in_idx)`**
Look-up embedding: `(B, T)` -> `(B, T, C)` = `(2, 1024, 768)`
Ogni intero viene sostituito dal suo vettore embedding.

**`x = self.drop_emb(tok_embeds)`**
Dropout sugli embedding: `(B, T, C)` shape invariata.
Durante il training, casualmente azzera alcune componenti degli embedding.

**`x = self.trf_blocks(x)`**
Applica i 12 `TransformerBlock4` in sequenza: `(B, T, C)` -> `(B, T, C)`.
La posizione e' codificata da RoPE dentro ogni blocco.

**`x = self.final_norm(x)`**
RMSNorm finale: `(B, T, C)` shape invariata.
Normalizza le rappresentazioni finali prima della proiezione vocab.

**`logits = self.out_head(x)`**
`(B, T, C)` -> `(B, T, vocab_size)` = `(2, 1024, 50257)`

**Logit vs Probabilita':**
- Durante il **training**: la cross-entropy loss applica softmax internamente
- Durante l'**inferenza**: `softmax(logits[:, -1, :])` per il prossimo token
- Restituire logit (non probabilita') e' piu' efficiente numericamente

**Output: `logits` shape `(B, T, vocab_size)`**
Punteggio grezzo per ogni token del vocabolario, per ogni posizione.


In [ ]:
# Capitolo 8: Demo GPT4Model — forward pass e conteggio parametri
import math, sys, torch, torch.nn as nn, torch.nn.functional as F

# (Classi RMSNorm, GQA, SwiGLU, TransformerBlock4 come definite prima)
# Qui usiamo versioni compatte

class RMSNorm(nn.Module):
    def __init__(self,d,eps=1e-5):
        super().__init__(); self.eps=eps; self.scale=nn.Parameter(torch.ones(d))
    def forward(self,x):
        return self.scale*(x/torch.sqrt(x.pow(2).mean(-1,keepdim=True)+self.eps))

def precompute_rope_freqs(hd,ctx,base=10000):
    t=1./(base**(torch.arange(0,hd,2).float()/hd))
    p=torch.arange(ctx,dtype=torch.float32); f=torch.outer(p,t)
    return torch.cos(f),torch.sin(f)

def apply_rope(x,cos,sin):
    _,_,T,D=x.shape; h=D//2; x1,x2=x[...,:h],x[...,h:]
    c=cos[:T].unsqueeze(0).unsqueeze(0); s=sin[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c-x2*s,x2*c+x1*s],-1)

class GQA(nn.Module):
    def __init__(self,di,do_,ctx,dp,nh,nk,rb=10000,qb=False):
        super().__init__()
        self.nh=nh;self.nk=nk;self.nr=nh//nk;self.hd=do_//nh;self.do_=do_
        self.wq=nn.Linear(di,do_,bias=qb);self.wk=nn.Linear(di,nk*self.hd,bias=qb)
        self.wv=nn.Linear(di,nk*self.hd,bias=qb);self.wo=nn.Linear(do_,do_,bias=False)
        self.drop=nn.Dropout(dp)
        self.register_buffer('mask',torch.triu(torch.ones(ctx,ctx),diagonal=1))
        c,s=precompute_rope_freqs(self.hd,ctx,rb)
        self.register_buffer('rc',c);self.register_buffer('rs',s)
    def forward(self,x):
        B,T,_=x.shape
        q=self.wq(x).view(B,T,self.nh,self.hd).transpose(1,2)
        k=self.wk(x).view(B,T,self.nk,self.hd).transpose(1,2)
        v=self.wv(x).view(B,T,self.nk,self.hd).transpose(1,2)
        q=apply_rope(q,self.rc,self.rs);k=apply_rope(k,self.rc,self.rs)
        k=k.repeat_interleave(self.nr,1);v=v.repeat_interleave(self.nr,1)
        sc=q@k.transpose(2,3);sc.masked_fill_(self.mask.bool()[:T,:T],-torch.inf)
        w=self.drop(torch.softmax(sc/self.hd**.5,dim=-1))
        return self.wo((w@v).transpose(1,2).reshape(B,T,self.do_))

class FFNSwiGLU(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        C=cfg['emb_dim'];H=int(math.ceil(8/3*C/256)*256)
        self.g=nn.Linear(C,H,bias=False);self.u=nn.Linear(C,H,bias=False)
        self.d=nn.Linear(H,C,bias=False)
    def forward(self,x): return self.d(F.silu(self.g(x))*self.u(x))

class TBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.att=GQA(cfg['emb_dim'],cfg['emb_dim'],cfg['context_length'],
            cfg['drop_rate'],cfg['n_heads'],cfg['n_kv_heads'],
            cfg.get('rope_base',10000),cfg.get('qkv_bias',False))
        self.ff=FFNSwiGLU(cfg);self.n1=RMSNorm(cfg['emb_dim']);self.n2=RMSNorm(cfg['emb_dim'])
        self.drop=nn.Dropout(cfg['drop_rate'])
    def forward(self,x):
        s=x;x=self.drop(self.att(self.n1(x)))+s
        s=x;x=self.drop(self.ff(self.n2(x)))+s
        return x

class GPT4Model(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.tok_emb=nn.Embedding(cfg['vocab_size'],cfg['emb_dim'])
        self.drop_emb=nn.Dropout(cfg['drop_rate'])
        self.trf_blocks=nn.Sequential(*[TBlock(cfg) for _ in range(cfg['n_layers'])])
        self.final_norm=RMSNorm(cfg['emb_dim'])
        self.out_head=nn.Linear(cfg['emb_dim'],cfg['vocab_size'],bias=False)
        self.context_length=cfg['context_length']
    def forward(self,idx):
        x=self.drop_emb(self.tok_emb(idx))
        x=self.trf_blocks(x)
        return self.out_head(self.final_norm(x))

GPT4_CONFIG_SMALL = {
    'vocab_size':50257,'context_length':256,'n_layers':12,
    'n_heads':12,'n_kv_heads':4,'emb_dim':768,
    'drop_rate':0.1,'qkv_bias':False,'rope_base':10000
}

model = GPT4Model(GPT4_CONFIG_SMALL)
model.eval()

# Conteggio parametri
def count_params(m): return sum(p.numel() for p in m.parameters())

print('Parametri per componente:')
print(f'  tok_emb:     {count_params(model.tok_emb):>12,}')
print(f'  trf_blocks:  {count_params(model.trf_blocks):>12,}')
for i in range(2):
    b=model.trf_blocks[i]
    print(f'    block[{i}].att: {count_params(b.att):>10,}')
    print(f'    block[{i}].ff:  {count_params(b.ff):>10,}')
print(f'  final_norm:  {count_params(model.final_norm):>12,}')
print(f'  out_head:    {count_params(model.out_head):>12,}')
print(f'  TOTALE:      {count_params(model):>12,}')
print(f'  In milioni:  {count_params(model)/1e6:.2f}M')

# Forward pass
idx = torch.randint(0, 50257, (2, 64))
with torch.no_grad():
    logits = model(idx)
print(f'\nForward pass:')
print(f'  Input:   {idx.shape} (B=2, T=64 token IDs)')
print(f'  Output:  {logits.shape} (B=2, T=64, vocab=50257 logits)')
print(f'  Ultimo token logits[0,-1]: min={logits[0,-1].min():.3f}, max={logits[0,-1].max():.3f}')


---
# Capitolo 9: GPT4_CONFIG_SMALL

## 9.1 Il Dizionario di Configurazione

```python
GPT4_CONFIG_SMALL = {
    'vocab_size':      50257,
    'context_length':  1024,
    'n_layers':        12,
    'n_heads':         12,
    'n_kv_heads':      4,
    'emb_dim':         768,
    'drop_rate':       0.1,
    'qkv_bias':        False,
    'rope_base':       10000,
}
```

Ogni chiave controlla un aspetto preciso del modello.


## 9.2 Analisi di Ogni Chiave

### `vocab_size = 50257`

Il numero di token unici nel vocabolario. GPT-2 usa BPE (Byte Pair Encoding) con tiktoken, che produce esattamente 50,257 token:
- 50,000 BPE merges
- 256 byte-level tokens di base
- 1 token speciale `<|endoftext|>`

Perche' 50,257 e non 50,000? I 256 byte-level tokens + 1 speciale = 50,257.

### `context_length = 1024`

La lunghezza massima della sequenza che il modello puo' processare in un singolo forward pass.
- GPT-2: 1024 token
- GPT-3: 2048 token
- LLaMA 2: 4096 token
- LLaMA 3: 128,000 token (con RoPE base=500000)

Con RoPE, il modello puo' essere esteso a lunghezze maggiori durante l'inferenza aumentando `rope_base` (YaRN, RoPE scaling techniques).

### `n_layers = 12`

Numero di blocchi Transformer sovrapposti. Come GPT-2 Small.
La profondita' determina la capacita' del modello di comporre trasformazioni complesse.

**Scaling con la profondita':**
| Modello | n_layers | Capacita' |
|---------|----------|----------|
| GPT-2 Small | 12 | Bassa |
| GPT-2 Medium | 24 | Media |
| GPT-2 XL | 48 | Alta |
| LLaMA 3 70B | 80 | Molto alta |

### `n_heads = 12`

Numero di teste Q nell'attenzione multi-testa. Con `emb_dim=768`:
`head_dim = 768/12 = 64`

Ogni testa lavora su uno spazio di 64 dimensioni, specializzandosi su diversi tipi di relazioni (es. testa 0: vicinanza sintattica, testa 5: coreference, etc.)

### `n_kv_heads = 4`

Numero di teste KV per GQA. Con `n_heads=12` e `n_kv_heads=4`:
- `n_rep = 12/4 = 3`: ogni KV head condiviso da 3 Q heads
- KV-cache ridotta di **3x** rispetto a MHA
- I 4 KV heads formano 4 'gruppi' di 3 Q heads ciascuno

### `emb_dim = 768`

La dimensione degli embedding e di tutti i vettori nel modello. Identico a GPT-2 Small. Determina la 'larghezza' del modello.
Tutti i layer hanno input/output di dimensione 768 (eccetto l'interno del FFN).

### `drop_rate = 0.1`

Probabilita' di dropout. Il 10% dei neuroni viene azzerato casualmente durante il training.
Usato in: embedding dropout, attention dropout, shortcut dropout.
Durante l'inferenza (`model.eval()`), il dropout e' disabilitato.

### `qkv_bias = False`

Nessun termine di bias nelle proiezioni Q, K, V. Convenzione LLaMA.
I bias aggiungono parametri ma raramente migliorano la qualita' in modelli grandi.

### `rope_base = 10000`

Il valore base per le frequenze RoPE: `theta_i = 1/base^(2i/D)`.
- `base=10000`: frequenze da 1.0 a ~0.00015 per head_dim=64
- Aumentare base -> frequenze piu' basse -> maggiore range posizionale
- LLaMA 3 usa `base=500000` per supportare 128K token di contesto


In [ ]:
# Capitolo 9: Analisi della configurazione
import math

cfg = {
    'vocab_size': 50257, 'context_length': 1024, 'n_layers': 12,
    'n_heads': 12, 'n_kv_heads': 4, 'emb_dim': 768,
    'drop_rate': 0.1, 'qkv_bias': False, 'rope_base': 10000
}

C = cfg['emb_dim']
H = cfg['n_heads']
G = cfg['n_kv_heads']
D = C // H  # head_dim
L = cfg['n_layers']
T = cfg['context_length']
V = cfg['vocab_size']
H_ff = int(math.ceil(8/3*C/256)*256)  # hidden dim FFN

print('Valori derivati dalla configurazione:')
print(f'  head_dim = emb_dim/n_heads = {C}/{H} = {D}')
print(f'  n_rep = n_heads/n_kv_heads = {H}/{G} = {H//G}')
print(f'  FFN hidden_dim = ceil(8/3*{C}/256)*256 = {H_ff}')
print()
print('Parametri per componente (n_layers=12):')
tok = V * C
per_block_att = C*C + C*G*D + C*G*D + C*C  # Wq, Wk, Wv, Wo (senza bias)
per_block_ff  = C*H_ff + C*H_ff + H_ff*C    # gate, up, down (senza bias)
per_block_norm = C + C  # norm1.scale + norm2.scale
per_block = per_block_att + per_block_ff + per_block_norm
final_norm = C
out_head = C * V

print(f'  tok_emb:         {tok:>12,}  ({tok/1e6:.2f}M)')
print(f'  Per blocco att:  {per_block_att:>12,}')
print(f'    Wq: {C}x{C}={C*C:,}')
print(f'    Wk: {C}x{G*D}={C*G*D:,}')
print(f'    Wv: {C}x{G*D}={C*G*D:,}')
print(f'    Wo: {C}x{C}={C*C:,}')
print(f'  Per blocco FF:   {per_block_ff:>12,}')
print(f'    gate+up+down: {C}x{H_ff}*2+{H_ff}x{C}={per_block_ff:,}')
print(f'  Per blocco norm: {per_block_norm:>12,}')
print(f'  Per blocco tot:  {per_block:>12,}')
print(f'  Tutti i blocchi: {L*per_block:>12,}  ({L*per_block/1e6:.2f}M)')
print(f'  final_norm:      {final_norm:>12,}')
print(f'  out_head:        {out_head:>12,}  ({out_head/1e6:.2f}M)')
total = tok + L*per_block + final_norm + out_head
print(f'  TOTALE:          {total:>12,}  ({total/1e6:.2f}M)')


## 9.3 Come Scalare la Configurazione

| Configurazione | n_layers | n_heads | n_kv_heads | emb_dim | Param |
|----------------|----------|---------|------------|---------|-------|
| Small (= GPT-2 S) | 12 | 12 | 4 | 768 | ~124M |
| Medium (= GPT-2 M) | 24 | 16 | 8 | 1024 | ~350M |
| Large (= GPT-2 L) | 36 | 20 | 5 | 1280 | ~774M |
| XL (= GPT-2 XL) | 48 | 25 | 5 | 1600 | ~1.5B |

**Regole di scaling:**
- `n_heads` deve dividere `emb_dim` -> `head_dim = emb_dim/n_heads`
- `n_kv_heads` deve dividere `n_heads`
- `emb_dim` multiplo di 64 (idealmente 128 o 256)
- Chinchilla scaling: parametri e dati devono crescere proporzionalmente


---
# Capitolo 10: Confronto Architetturale Completo

## 10.1 GPT-2 vs GPT-4 Style: Tabella Dettagliata

| Componente | GPT-2 Small | gpt_model4.py |
|------------|-------------|---------------|
| Normalizzazione | LayerNorm (mu + sigma) | RMSNorm (solo sigma) |
| Posizione | pos_emb appreso, T*C=786K param | RoPE, 0 param |
| Tipo attenzione | MHA (H=K=V=12 heads) | GQA (Q=12, KV=4 heads) |
| FFN attivazione | GELU | SwiGLU |
| FFN espansione | 4x (3072) | 8/3x (2048) |
| FFN matrici | 2 | 3 |
| Bias in QKV | Si' | No |
| Bias in FFN | Si' | No |
| n_layers | 12 | 12 |
| n_heads | 12 | 12 |
| emb_dim | 768 | 768 |
| vocab_size | 50257 | 50257 |
| Parametri totali | ~124M | ~124M |

## 10.2 Impatto sulle Performance

| Modifica | Impatto Training | Impatto Inferenza | Impatto Qualita' |
|----------|-----------------|-------------------|------------------|
| RMSNorm | +15% velocita' | +15% velocita' | Identica |
| RoPE | Nessuno | Nessuno | +++ generalizzazione |
| GQA (H/G=3) | -3% param Wk/Wv | -3x KV-cache | -0.5-1pp perplexity |
| SwiGLU | +5% compute | +5% compute | +1-2pp perplexity |


## 10.3 Confronto con LLaMA 1, LLaMA 3, Mistral 7B

| Modello | n_layers | n_heads | n_kv_heads | emb_dim | FFN hidden | Param |
|---------|----------|---------|------------|---------|------------|-------|
| LLaMA 1 7B | 32 | 32 | 32 | 4096 | 11008 | 6.7B |
| LLaMA 2 7B | 32 | 32 | 32 | 4096 | 11008 | 6.7B |
| LLaMA 2 70B | 80 | 64 | 8 | 8192 | 28672 | 69.6B |
| LLaMA 3 8B | 32 | 32 | 8 | 4096 | 14336 | 8.0B |
| Mistral 7B | 32 | 32 | 8 | 4096 | 14336 | 7.2B |
| Gemma 7B | 28 | 16 | 16 | 3072 | 24576 | 8.5B |
| **gpt_model4.py** | **12** | **12** | **4** | **768** | **2048** | **~124M** |

**Osservazioni:**
- Tutti i moderni LLM convergono verso RMSNorm + RoPE + GQA + SwiGLU
- LLaMA 1 usa MHA (n_kv_heads = n_heads), LLaMA 2/3 e Mistral usano GQA
- `gpt_model4.py` e' una versione 'tascabile' di queste architetture
- Sostituendo la config, diventa un modello production-ready


In [ ]:
# Capitolo 10: Confronto architetturale con conteggio parametri
import math

def count_params(V, C, L, H, G, T_ctx, rope=True):
    D = C // H
    H_ff = int(math.ceil(8/3*C/256)*256)
    tok = V * C
    pos = 0 if rope else T_ctx * C
    att = C*C + C*G*D + C*G*D + C*C  # Wq, Wk, Wv, Wo
    ff = C*H_ff + C*H_ff + H_ff*C  # gate, up, down
    norm = C * 2  # norm1 + norm2 per blocco
    blocks = L * (att + ff + norm)
    out = C * V
    return tok + pos + blocks + C + out  # +C per final_norm

modelli = [
    ('LLaMA 1 7B',   32000, 4096, 32, 32, 32, 2048),
    ('LLaMA 2 7B',   32000, 4096, 32, 32, 32, 4096),
    ('LLaMA 2 70B',  32000, 8192, 80, 64,  8, 4096),
    ('LLaMA 3 8B',  128256, 4096, 32, 32,  8, 8192),
    ('Mistral 7B',   32000, 4096, 32, 32,  8, 4096),
    ('gpt_model4.py',50257,  768, 12, 12,  4, 1024),
]

print(f'{"Modello":<20} {"Param (M)":>10} {"head_dim":>9} {"n_rep":>6}')
print('-' * 50)
for nome, V, C, L, H, G, Tctx in modelli:
    p = count_params(V, C, L, H, G, Tctx)
    hd = C // H
    nr = H // G
    print(f'{nome:<20} {p/1e6:>10.1f} {hd:>9} {nr:>6}')


---
# Capitolo 11: Diagramma del Flusso Dati Completo

## 11.1 Flusso dall'Input ai Logits

Seguiamo un batch di dimensione B=2, T=6 token attraverso l'intero modello.

```
INPUT: in_idx shape (2, 6)
  [[464, 995, 318, 281, 1672, 13],   # 'The dog is an animal .'
   [1212, 318, 257, 1332,   0,  0]]   # 'This is a test'

STEP 1: tok_emb
  (2, 6) -> (2, 6, 768)
  Ogni intero sostituito da vettore embedding di 768 dim

STEP 2: drop_emb
  (2, 6, 768) -> (2, 6, 768)
  10% dei valori azzerati (solo training)

STEP 3: TransformerBlock4 x 12
  (2, 6, 768) -> (2, 6, 768)  [shape invariata]

  Dentro ogni blocco:
  +--- shortcut = x  (2,6,768) ---+
  | norm1 (RMSNorm)  (2,6,768)    |
  | att (GQA+RoPE):               |
  |   W_query: (2,6,768)->(2,12,6,64) Q
  |   W_key:   (2,6,768)->(2,4,6,64)  K
  |   W_value: (2,6,768)->(2,4,6,64)  V
  |   apply_rope(Q): (2,12,6,64) rotato
  |   apply_rope(K): (2,4,6,64)  rotato
  |   expand K->: (2,12,6,64) (x3 interleave)
  |   expand V->: (2,12,6,64) (x3 interleave)
  |   Q@K.T: (2,12,6,6) attn scores
  |   mask+softmax/sqrt(64): (2,12,6,6)
  |   @V: (2,12,6,64)->reshape->(2,6,768)
  |   out_proj: (2,6,768)
  | drop_shortcut: (2,6,768)          |
  +-- + shortcut --(2,6,768) ----------+

  +--- shortcut = x  (2,6,768) ---+
  | norm2 (RMSNorm)  (2,6,768)    |
  | ff (SwiGLU):                  |
  |   gate: (2,6,768)->(2,6,2048) |
  |   SiLU: (2,6,2048)            |
  |   up:   (2,6,768)->(2,6,2048) |
  |   gate*up: (2,6,2048)         |
  |   down: (2,6,2048)->(2,6,768) |
  | drop_shortcut: (2,6,768)      |
  +-- + shortcut --(2,6,768) -----+

STEP 4: final_norm (RMSNorm)
  (2, 6, 768) -> (2, 6, 768)

STEP 5: out_head (Linear)
  (2, 6, 768) -> (2, 6, 50257)

OUTPUT: logits shape (2, 6, 50257)
  Per generare il prossimo token: logits[:, -1, :] -> (2, 50257)
  Poi: softmax -> probabilita', argmax o sampling -> token ID
```


In [ ]:
# Capitolo 11: Visualizzazione del flusso con shapes reali
import math, torch, torch.nn as nn, torch.nn.functional as F

# Setup minimo per tracciare le shapes
B, T, C = 2, 6, 768
H, G, L = 12, 4, 12
V = 50257
D = C // H  # 64
H_ff = int(math.ceil(8/3*C/256)*256)  # 2048

print('Flusso dati in GPT4Model - shapes ad ogni step:')
print(f'Configurazione: B={B}, T={T}, C={C}, H={H}, G={G}')
print()

steps = [
    ('Input in_idx',     f'({B}, {T})',          'token IDs interi'),
    ('tok_emb',          f'({B}, {T}, {C})',     'lookup embedding'),
    ('drop_emb',         f'({B}, {T}, {C})',     'dropout (training)'),
    ('-- BLOCCO 1..12 --','',                    ''),
    ('  norm1 (RMSNorm)',f'({B}, {T}, {C})',     'normalizza x'),
    ('  W_query',        f'({B}, {H}, {T}, {D})','query heads'),
    ('  W_key',          f'({B}, {G}, {T}, {D})','key heads (GQA)'),
    ('  W_value',        f'({B}, {G}, {T}, {D})','value heads (GQA)'),
    ('  apply_rope(Q)',  f'({B}, {H}, {T}, {D})','rotazione pos Q'),
    ('  apply_rope(K)',  f'({B}, {G}, {T}, {D})','rotazione pos K'),
    ('  expand K->H',    f'({B}, {H}, {T}, {D})','repeat_interleave x{H//G}'),
    ('  expand V->H',    f'({B}, {H}, {T}, {D})','repeat_interleave x{H//G}'),
    ('  Q@K.T',          f'({B}, {H}, {T}, {T})','attn scores'),
    ('  softmax',        f'({B}, {H}, {T}, {T})','pesi norm.'),
    ('  @V + out_proj',  f'({B}, {T}, {C})',     'context vector'),
    ('  + shortcut',     f'({B}, {T}, {C})',     'residual 1'),
    ('  norm2 (RMSNorm)',f'({B}, {T}, {C})',     'norm prima FFN'),
    ('  FFN gate+up',    f'({B}, {T}, {H_ff})',  'espansione 8/3x'),
    ('  SiLU * up',      f'({B}, {T}, {H_ff})',  'gating'),
    ('  FFN down',       f'({B}, {T}, {C})',     'proiezione ritorno'),
    ('  + shortcut',     f'({B}, {T}, {C})',     'residual 2'),
    ('-- FINE BLOCCO --', '',                    ''),
    ('final_norm',       f'({B}, {T}, {C})',     'RMSNorm finale'),
    ('out_head',         f'({B}, {T}, {V})',     'logits vocab'),
    ('OUTPUT logits',    f'({B}, {T}, {V})',     'punteggi non normalizzati'),
]

for step, shape, note in steps:
    if step.startswith('--'):
        print(f'  {step}')
    else:
        print(f'  {step:<22} {shape:<20} # {note}')


---
# Appendice A: Matematica del Self-Attention

## A.1 Derivazione Completa

Deriviamo formalmente perche' l'attenzione scaled dot-product funziona.

### Setup

Data una sequenza X di T vettori, ciascuno di dimensione C:
```
X in R^(T x C)
```

Le proiezioni lineari:
```
Q = X @ W_Q    W_Q in R^(C x d_k)    Q in R^(T x d_k)
K = X @ W_K    W_K in R^(C x d_k)    K in R^(T x d_k)
V = X @ W_V    W_V in R^(C x d_v)    V in R^(T x d_v)
```

### Il Prodotto QK^T

```
A = Q @ K^T    A in R^(T x T)
A[i,j] = Q[i,:] . K[j,:] = sum_k Q[i,k] * K[j,k]
```

A[i,j] misura la 'compatibilita'' tra la query in posizione i e la chiave in posizione j.

### Scaling per sqrt(d_k)

Se le componenti di Q e K sono iid con media 0 e varianza 1:
```
Var(Q[i,:] . K[j,:]) = sum_k Var(Q[i,k] * K[j,k]) = sum_k 1 = d_k
```

Quindi std(A[i,j]) = sqrt(d_k). Dividendo per sqrt(d_k):
```
Var(A[i,j]/sqrt(d_k)) = d_k / d_k = 1
```

Questo mantiene i valori nella zona attiva del softmax.

### Softmax e Normalizzazione

```
alpha = softmax(A/sqrt(d_k), dim=-1)
alpha[i,:] = exp(A[i,:]/sqrt(d_k)) / sum_j exp(A[i,j]/sqrt(d_k))
```

alpha[i,:] e' un vettore di probabilita' (somma a 1), che rappresenta quanta attenzione la posizione i presta a ogni altra posizione j.

### Output come Media Pesata

```
Attn(Q,K,V) = alpha @ V
Attn(Q,K,V)[i,:] = sum_j alpha[i,j] * V[j,:]
```

L'output per la posizione i e' una media pesata dei valori, con pesi dati dalla compatibilita' Q[i]-K[j].

### Complessita' Computazionale

- `Q @ K^T`: O(T^2 * d_k) operazioni
- `softmax`: O(T^2)
- `alpha @ V`: O(T^2 * d_v)
- **Totale**: O(T^2 * D) per T tokens, D dimensioni

Per T=1024 e D=64: 1024^2 * 64 ≈ 67M operazioni **per testa per layer**.
Con H=12 teste e L=12 layer: ~9.6 miliardi di operazioni solo per l'attenzione.

### Flash Attention

Flash Attention (Dao et al., 2022) riorganizza il calcolo per sfruttare la gerarchia della memoria GPU (SRAM vs HBM), riducendo la complessita' in **memoria** da O(T^2) a O(T) senza cambiare il risultato matematico.


In [ ]:
# Appendice A: Implementazione manuale di Attention
import torch
import torch.nn.functional as F

def manual_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    # Passo 1: Scaled dot product
    scores = Q @ K.transpose(-2, -1) / d_k**0.5  # (T, T)
    # Passo 2: Maschera causale
    if mask is not None:
        scores = scores.masked_fill(mask, -torch.inf)
    # Passo 3: Softmax
    weights = torch.softmax(scores, dim=-1)
    # Passo 4: Aggregazione
    return weights @ V, weights

torch.manual_seed(42)
T, d_k, d_v = 5, 4, 4
Q = torch.randn(T, d_k)
K = torch.randn(T, d_k)
V = torch.randn(T, d_v)

# Maschera causale
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)

output, weights = manual_attention(Q, K, V, mask)

print(f'Q shape: {Q.shape}')
print(f'K shape: {K.shape}')
print(f'V shape: {V.shape}')
print(f'Output shape: {output.shape}')
print(f'Pesi attenzione shape: {weights.shape}')
print()
print('Pesi di attenzione (ogni riga somma a 1):')
for i in range(T):
    row = weights[i].detach().numpy().round(3)
    print(f'  pos {i}: {row}  (sum={weights[i].sum():.3f})')
print()
print('Nota: la triangolarita inferiore garantisce che pos i vede solo pos 0..i')
print('(i valori 0 sopra la diagonale vengono da -inf -> exp(-inf) = 0)')


---
# Appendice B: Considerazioni sull'Addestramento

## B.1 Cross-Entropy Loss per Language Modeling

L'obiettivo del language modeling e' massimizzare la probabilita' del token corretto data la sequenza precedente.

Per un token target t con probabilita' p_t:
```
loss = -log(p_t) = -log(softmax(logits)[t])
```

In PyTorch, `nn.CrossEntropyLoss` combina softmax e log-loss:
```python
loss = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
```

**Perplexity:** misura comune per LM
```
PPL = exp(loss)   (piu bassa e meglio)
```

Un modello che assegna probabilita' uniforme su 50257 token ha PPL = 50257.
GPT-2 Small su WebText raggiunge PPL ≈ 29.4.

## B.2 AdamW Optimizer

AdamW (Loshchilov & Hutter, 2019) e' il standard per addestrare LLM:

```
m_t = beta1 * m_{t-1} + (1-beta1) * g_t       # momento del primo ordine
v_t = beta2 * v_{t-1} + (1-beta2) * g_t^2    # momento del secondo ordine
m_hat = m_t / (1 - beta1^t)
v_hat = v_t / (1 - beta2^t)
theta = theta - lr * m_hat / (sqrt(v_hat) + eps) - lr * wd * theta  # weight decay
```

- `beta1 = 0.9`: coefficiente di decadimento per il momento
- `beta2 = 0.999`: coefficiente per la varianza stimata
- `eps = 1e-8`: stabilita' numerica
- `wd = 0.1`: weight decay (L2 regularization, ma corretto)

**AdamW vs Adam:** AdamW applica il weight decay direttamente ai parametri invece di aggiungerlo al gradiente. E' matematicamente piu' corretto.

## B.3 Learning Rate Schedule

I moderni LLM usano un schedule in due fasi:

**1. Warmup lineare (tipicamente 2000-4000 step):**
```
lr(t) = lr_max * t / warmup_steps
```

Il warmup evita instabilita' all'inizio quando i gradienti sono grandi.

**2. Cosine decay:**
```
lr(t) = lr_min + 0.5*(lr_max-lr_min) * (1 + cos(pi * t/T_total))
```

Il learning rate decresce smoothly fino a lr_min ≈ lr_max/10.

**Valori tipici per ~124M parametri:**
- `lr_max = 6e-4` (come GPT-2 Small)
- `warmup_steps = 2000`
- `lr_min = 6e-5`

## B.4 Gradient Clipping

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

Limita la norma del gradiente a max_norm=1.0. Previene esplosioni del gradiente che potrebbero destabilizzare il training (specialmente all'inizio).

## B.5 Mixed Precision Training

I moderni LLM vengono addestrati con bf16 (bfloat16):
- 16 bit invece di 32: 2x meno memoria, 2x piu' veloce su Tensor Cores
- bf16 ha range piu' grande di fp16 (meno overflow/underflow)
- Parametri e gradienti master in fp32 per la stabilita'

```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    logits = model(idx)
    loss = F.cross_entropy(logits.view(-1, V), targets.view(-1))
```


In [ ]:
# Appendice B: Simulazione del training loop
import torch
import torch.nn.functional as F

def cosine_lr_schedule(step, warmup_steps, total_steps, lr_max, lr_min):
    import math
    if step < warmup_steps:
        return lr_max * step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

# Visualizza il learning rate schedule
lr_max, lr_min = 6e-4, 6e-5
warmup = 100
total = 1000
steps = [0, 50, 100, 200, 400, 600, 800, 1000]

print('Learning Rate Schedule (warmup + cosine decay):')
print(f'  lr_max={lr_max}, lr_min={lr_min}, warmup={warmup}, total={total}')
print()
print(f'  {"Step":>6}  {"LR":>12}  Fase')
print(f'  {"------":>6}  {"------------":>12}  ----')
for s in steps:
    lr = cosine_lr_schedule(s, warmup, total, lr_max, lr_min)
    fase = 'warmup' if s < warmup else 'cosine'
    print(f'  {s:>6}  {lr:>12.2e}  {fase}')

# Demo loss e perplexity
print()
print('Relazione Loss -> Perplexity:')
import math
for loss in [10.8, 5.0, 3.5, 3.0, 2.5, 2.0]:
    ppl = math.exp(loss)
    print(f'  loss={loss:.1f} -> PPL={ppl:.1f}')
print('  (loss=log(50257)=10.82 -> PPL=50257: modello random)')
print('  (GPT-2 Small su WebText: loss≈3.38 -> PPL≈29.4)')


---
# Appendice C: Inferenza Autoregressiva

## C.1 Generazione Token per Token

Durante l'inferenza, il modello genera testo un token alla volta in modo autoregressivo:

```python
def generate(model, prompt_ids, max_new_tokens, temperature=1.0, top_k=50):
    x = prompt_ids.clone()
    for _ in range(max_new_tokens):
        # Tronca al context_length se necessario
        x_ctx = x[:, -model.context_length:]
        # Forward pass
        with torch.no_grad():
            logits = model(x_ctx)
        # Considera solo l'ultimo token
        logits_last = logits[:, -1, :]  # (B, V)
        # Sampling
        next_token = sample(logits_last, temperature, top_k)
        # Aggiungi il nuovo token
        x = torch.cat([x, next_token], dim=1)
    return x
```

## C.2 Strategie di Sampling

### Greedy Decoding
```python
next_token = logits.argmax(dim=-1, keepdim=True)
```

Sempre sceglie il token piu' probabile. Deterministico ma tende a output ripetitivi.

### Temperature Sampling
```python
probs = torch.softmax(logits / temperature, dim=-1)
next_token = torch.multinomial(probs, 1)
```

- `temperature > 1`: distribuzione piu' uniforme, output piu' diversificato
- `temperature < 1`: distribuzione piu' peaked, output piu' conservativo
- `temperature -> 0`: diventa greedy decoding

### Top-K Sampling
```python
topk_vals, topk_idx = logits.topk(k=50)
# Azzera i logit al di fuori del top-k
new_logits = torch.full_like(logits, -torch.inf)
new_logits.scatter_(1, topk_idx, topk_vals)
probs = torch.softmax(new_logits / temperature, dim=-1)
next_token = torch.multinomial(probs, 1)
```

### Top-P (Nucleus) Sampling
Seleziona il sottoinsieme piu' piccolo di token le cui probabilita' sommate superano p.
Adatta dinamicamente il numero di token considerati.

## C.3 KV-Cache nell'Inferenza

Con la KV-cache, invece di ricalcolare K e V per tutti i token precedenti:
1. Al primo step: calcola K, V per tutti i T_prompt token, salvali
2. A ogni step successivo: calcola K, V solo per il nuovo token, appendili alla cache
3. L'attenzione usa tutti i K, V accumulati nella cache

**Complessita' senza cache:** O(T^2) operazioni per step
**Complessita' con cache:** O(T) operazioni per step

GQA riduce la dimensione della cache di H/G = 3x.


In [ ]:
# Appendice C: Demo di inferenza autoregressiva
import torch
import torch.nn.functional as F

def sample_top_k(logits, k=50, temperature=1.0):
    if temperature != 1.0:
        logits = logits / temperature
    if k > 0:
        vals, idx = logits.topk(k, dim=-1)
        new_logits = torch.full_like(logits, float('-inf'))
        new_logits.scatter_(-1, idx, vals)
        logits = new_logits
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

# Demo con logits casuali (senza un modello reale)
torch.manual_seed(42)
V = 100  # vocabolario piccolo per demo
logits = torch.randn(1, V)

print('Demo sampling con vocabolario di 100 token:')
print()

# Greedy
greedy = logits.argmax(dim=-1).item()
print(f'Greedy: token {greedy} (prob={F.softmax(logits,-1)[0,greedy]:.3f})')

# Temperature sampling
print()
print('Temperature sampling (10 campioni):')
for temp in [0.5, 1.0, 2.0]:
    samples = [sample_top_k(logits, k=0, temperature=temp).item() for _ in range(5)]
    print(f'  T={temp}: {samples}')

# Top-k
print()
print('Top-k sampling (k=5, T=1.0, 10 campioni):')
vals, idx = logits.topk(5)
print(f'  Top-5 token IDs: {idx[0].tolist()}')
print(f'  Top-5 probs: {F.softmax(vals, -1)[0].numpy().round(3).tolist()}')
samples = [sample_top_k(logits, k=5, temperature=1.0).item() for _ in range(10)]
print(f'  Campioni: {samples}')


---
# Appendice D: Confronto con Modelli Reali

## D.1 LLaMA 3.1 8B Configuration

```python
LLAMA3_8B_CONFIG = {
    'vocab_size': 128256,    # tiktoken tokenizer
    'context_length': 131072, # 128K token con RoPE scaling
    'n_layers': 32,
    'n_heads': 32,
    'n_kv_heads': 8,         # GQA: n_rep = 32/8 = 4
    'emb_dim': 4096,
    'drop_rate': 0.0,        # no dropout in produzione
    'qkv_bias': False,
    'rope_base': 500000,     # alto per contesti lunghi
}
# head_dim = 4096/32 = 128
# FFN hidden = ceil(8/3*4096/256)*256 = 14336
# Parametri: ~8B
```

## D.2 Mistral 7B Configuration

```python
MISTRAL_7B_CONFIG = {
    'vocab_size': 32000,
    'context_length': 4096,  # sliding window 4096 token
    'n_layers': 32,
    'n_heads': 32,
    'n_kv_heads': 8,         # GQA
    'emb_dim': 4096,
    'drop_rate': 0.0,
    'qkv_bias': False,
    'rope_base': 10000,
}
# Usa Sliding Window Attention (SWA) per contesti piu' lunghi
# Parametri: ~7.2B
```

## D.3 Gemma 7B Configuration

```python
GEMMA_7B_CONFIG = {
    'vocab_size': 256000,    # tokenizer piu' grande
    'context_length': 8192,
    'n_layers': 28,
    'n_heads': 16,
    'n_kv_heads': 16,        # MHA (non GQA per 7B)
    'emb_dim': 3072,
    'drop_rate': 0.0,
    'qkv_bias': False,
    'rope_base': 10000,
}
# Usa GeGLU invece di SwiGLU (GELU nel gate invece di SiLU)
# Parametri: ~8.5B
```

## D.4 Confronto con GPT4_CONFIG_SMALL

| Config | n_layers | n_heads | n_kv | emb_dim | context | Param |
|--------|----------|---------|------|---------|---------|-------|
| LLaMA 3 8B | 32 | 32 | 8 | 4096 | 128K | 8B |
| Mistral 7B | 32 | 32 | 8 | 4096 | 32K | 7B |
| Gemma 7B | 28 | 16 | 16 | 3072 | 8K | 8.5B |
| **gpt_model4.py** | **12** | **12** | **4** | **768** | **1K** | **124M** |

**La struttura e' identica** — solo le dimensioni cambiano.
Sostituendo `GPT4_CONFIG_SMALL` con configurazioni piu' grandi, `gpt_model4.py` diventa un LLM production-ready.


---
# Appendice E: Glossario Tecnico

| Termine | Definizione |
|---------|-------------|
| **Attention** | Meccanismo che calcola una media pesata dei valori, con pesi basati sulla compatibilita' tra query e key |
| **Autoregressive** | Modello che genera sequenze predicendo un token alla volta condizionato sui precedenti |
| **BF16 (bfloat16)** | Formato numerico a 16 bit con ampio range (come float32) ma bassa precisione |
| **BPE** | Byte Pair Encoding: algoritmo di tokenizzazione che fonde iterativamente le coppie piu' frequenti |
| **Buffer** | Tensore registrato in un nn.Module che viene salvato ma non aggiornato dall'optimizer |
| **Causal Mask** | Maschera triangolare superiore che impedisce al token i di vedere i token j > i |
| **Context Length** | Numero massimo di token che il modello puo' processare in un forward pass |
| **Cross-Entropy Loss** | -log(p_target): misura quanto il modello e' sorpreso dal token corretto |
| **Dropout** | Tecnica di regolarizzazione che azzera casualmente una frazione p di neuroni |
| **Embedding** | Vettore denso che rappresenta un token o una posizione discreta |
| **GELU** | Gaussian Error Linear Unit: x * Phi(x), smooth version di ReLU |
| **GQA** | Grouped Query Attention: H query heads ma G < H key/value heads |
| **Head Dim** | Dimensione di ogni testa di attenzione: head_dim = emb_dim / n_heads |
| **KV-Cache** | Memoria durante l'inferenza che salva K e V calcolati per i token passati |
| **LayerNorm** | Normalizzazione per esempio, usando media e varianza delle feature |
| **Logit** | Punteggio grezzo non normalizzato prima del softmax |
| **LLM** | Large Language Model: modello linguistico con miliardi di parametri |
| **MHA** | Multi-Head Attention: attenzione con H teste indipendenti |
| **MQA** | Multi-Query Attention: caso estremo di GQA con 1 solo KV head |
| **n_kv_heads** | Numero di teste per key e value in GQA |
| **n_rep** | Fattore di ripetizione in GQA: n_heads / n_kv_heads |
| **Parameter** | Tensore appreson aggiornato dall'optimizer durante il training |
| **Perplexity** | exp(cross_entropy_loss): misura standard per LM; piu' bassa e' meglio |
| **Pre-Norm** | Normalizzazione applicata prima del sublayer (not dopo) |
| **Residual Connection** | Skip connection: output = input + sublayer(input) |
| **RMSNorm** | Root Mean Square Norm: normalizza per RMS invece che per (mu, sigma) |
| **RoPE** | Rotary Position Embedding: codifica posizionale tramite rotazione di Q e K |
| **Scaled Dot-Product** | Attenzione con divisione per sqrt(d_k) per stabilita' del softmax |
| **SiLU** | Sigmoid Linear Unit: x * sigmoid(x), anche chiamata Swish |
| **SwiGLU** | Swish-Gated Linear Unit: SiLU(W_gate*x) * (W_up*x) |
| **Temperature** | Parametro di divisione dei logit prima del softmax: T>1 piu' casuale |
| **Tiktoken** | Tokenizer BPE di OpenAI usato da GPT-2 e GPT-4 |
| **Top-K** | Sampling considerando solo i K token piu' probabili |
| **Top-P** | Nucleus sampling: subset minimo con probabilita' cumulativa >= p |
| **Transformer** | Architettura basata su self-attention introdotta da Vaswani et al. 2017 |
| **Vanishing Gradient** | Fenomeno per cui i gradienti diventano esponenzialmente piccoli nelle reti profonde |
| **Vocab Size** | Numero di token unici nel vocabolario |
| **Weight Tying** | Condivisione dei pesi tra tok_emb e out_head |


In [ ]:
# Appendice E: Riepilogo finale - tutti i numeri chiave
import math

print('GPT4_CONFIG_SMALL - Tutti i Numeri Chiave')
print('=' * 55)

V, C, L, H, G, T_ctx = 50257, 768, 12, 12, 4, 1024
D = C // H
H_ff = int(math.ceil(8/3*C/256)*256)
n_rep = H // G

print(f'Vocabolario:         {V:>10,} token')
print(f'Dimensione emb:      {C:>10,}')
print(f'N. layer:            {L:>10}')
print(f'N. Q heads:          {H:>10}')
print(f'N. KV heads:         {G:>10} (GQA n_rep={n_rep})')
print(f'head_dim:            {D:>10}')
print(f'Context length:      {T_ctx:>10,} token')
print(f'FFN hidden_dim:      {H_ff:>10,} (8/3*C)')
print(f'rope_base:           {10000:>10}')

# Parametri
tok = V * C
att = C*C + C*G*D + C*G*D + C*C
ff  = C*H_ff + C*H_ff + H_ff*C
norm = C + C
block = att + ff + norm
final = C + C*V
total = tok + L*block + final

print()
print('Parametri:')
print(f'  tok_emb:             {tok:>12,}  ({tok/1e6:.1f}M)')
print(f'  Per blocco (att):    {att:>12,}')
print(f'  Per blocco (ff):     {ff:>12,}')
print(f'  Per blocco (norm):   {norm:>12,}')
print(f'  Per blocco (totale): {block:>12,}  ({block/1e6:.2f}M)')
print(f'  Tutti i blocchi:     {L*block:>12,}  ({L*block/1e6:.1f}M)')
print(f'  final_norm+out:      {final:>12,}  ({final/1e6:.1f}M)')
print(f'  TOTALE:              {total:>12,}  ({total/1e6:.1f}M)')

# KV-cache durante inferenza
kv_cache_per_token = G * D * 2 * L * 2  # bytes (float16)
kv_cache_1024 = kv_cache_per_token * T_ctx
print()
print(f'KV-cache (float16):')
print(f'  Per token:           {kv_cache_per_token:>12,} bytes')
print(f'  Per context T={T_ctx}: {kv_cache_1024:>12,} bytes ({kv_cache_1024/1024:.1f} KB)')
print(f'  (MHA sarebbe {H/G:.0f}x piu grande: {kv_cache_1024*H/G/1024:.1f} KB)')


## 3.6 Analisi Numerica: Stabilita' di RMSNorm

Vediamo come RMSNorm mantiene la stabilita' delle attivazioni su un forward pass profondo.

### Esperimento: Propagazione senza vs con normalizzazione

In una rete profonda senza normalizzazione, le attivazioni tendono a:
- **Esplodere**: se la norma dei pesi e' > 1 in media
- **Collassare**: se la norma dei pesi e' < 1 in media

Con RMSNorm, ogni layer 'reimposta' la scala delle attivazioni, garantendo che le informazioni possano fluire attraverso tutti i layer.

La formula:
```
x_norm = x / RMS(x)   =>   RMS(x_norm) ≈ 1
```

Il parametro scale (gamma) permette poi al modello di scegliere la scala ottimale per ogni feature, iniziando da gamma=1 (nessuna modifica).


In [ ]:
# Capitolo 3 extra: Propagazione delle attivazioni con e senza RMSNorm
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.scale*(x/torch.sqrt(x.pow(2).mean(-1,keepdim=True)+self.eps))

torch.manual_seed(0)
C = 64
L = 24  # molti layer per vedere l'effetto

# Inizializzazione tipica: N(0, 1/sqrt(C))
def make_linear(C):
    w = nn.Linear(C, C, bias=False)
    nn.init.normal_(w.weight, std=1.0/C**0.5)
    return w

# Senza normalizzazione
layers_no_norm = [make_linear(C) for _ in range(L)]
# Con RMSNorm
layers_with_norm = [(make_linear(C), RMSNorm(C)) for _ in range(L)]

x = torch.randn(1, C)
x_normed = x.clone()

print(f'Propagazione su {L} layer (dim={C})')
print(f'Input: RMS(x) = {x.pow(2).mean().sqrt().item():.4f}')
print()
print(f'{"Layer":>6}  {"No Norm RMS":>14}  {"Con Norm RMS":>14}')
print('-' * 40)
with torch.no_grad():
    for i, (layer_nn, (layer_wn, norm)) in enumerate(zip(layers_no_norm, layers_with_norm)):
        x = torch.relu(layer_nn(x))
        x_normed = norm(torch.relu(layer_wn(x_normed)))
        rms_nn = x.pow(2).mean().sqrt().item()
        rms_wn = x_normed.pow(2).mean().sqrt().item()
        if i in [0, 4, 8, 12, 16, 19, 20, 21, 22, 23]:
            print(f'{i+1:>6}  {rms_nn:>14.4f}  {rms_wn:>14.4f}')

print()
print('Con RMSNorm: RMS rimane vicino a 1 ad ogni layer')
print('Senza norma: RMS puo variare drasticamente (esplodere o collassare)')


## 4.8 Visualizzazione Numerica delle Frequenze RoPE

Le frequenze RoPE coprono uno spettro da alta a bassa frequenza:

- **Alta frequenza** (i piccolo): ogni posizione ha un angolo molto diverso
  Utile per distinguere posizioni vicine (1, 2, 3, ...)
- **Bassa frequenza** (i grande): gli angoli cambiano lentamente
  Utile per distinguere posizioni lontane (100, 200, 500, ...)

Questo e' analogo alla trasformata di Fourier: diverse frequenze codificano diverse scale di informazione posizionale.

### Numero di Giri Completi su context_length Posizioni

```
N_giri_i = theta_i * context_length / (2*pi)
         = (1/base^(2i/D)) * T / (2*pi)
```

- Coppia i=0: theta=1.0, T=1024 -> giri = 1024/(2*pi) ≈ 163 giri
- Coppia i=31: theta=0.00015, T=1024 -> giri ≈ 0.024 giri (quasi fermo)

Questa distribuzione multi-scala garantisce che il modello possa distinguere posizioni a qualsiasi distanza.


In [ ]:
# Capitolo 4 extra: Analisi spettrale delle frequenze RoPE
import torch
import math

head_dim = 64
base = 10000
T = 1024

theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))

print(f'Analisi frequenze RoPE (head_dim={head_dim}, base={base}, T={T})')
print()
print(f'{"Coppia i":>8}  {"theta_i":>12}  {"Periodo":>12}  {"Giri in T":>12}')
print('-' * 50)
for i in range(0, head_dim//2, 4):
    ti = theta[i].item()
    period = 2 * math.pi / ti if ti > 0 else float('inf')
    turns = T / period
    print(f'{i:>8}  {ti:>12.6f}  {period:>12.1f}  {turns:>12.3f}')

print()
print('Interpretazione:')
print('  - Coppia 0: si gira ~163 volte su T=1024 posizioni (distingue posizioni vicine)')
print('  - Coppia 31: quasi immobile su T=1024 (distingue posizioni lontane)')
print('  - Analogia: lancette di orologio a diverse velocita')

# Angoli per le prime 8 posizioni, prima coppia
print()
print('Angoli di rotazione per coppia i=0 (alta freq):')
for pos in range(8):
    angle = pos * theta[0].item()
    cos_val = math.cos(angle)
    sin_val = math.sin(angle)
    print(f'  pos={pos}: angle={angle:.3f} rad -> cos={cos_val:.3f}, sin={sin_val:.3f}')


## 5.7 Analisi Dettagliata del `repeat_interleave`

Il punto chiave di GQA e' come i KV heads vengono espansi per corrispondere ai Q heads.

**`torch.repeat_interleave(n_rep, dim)`** vs **`torch.repeat()`**:

```
tensor = [A, B, C, D]  # 4 elementi

repeat_interleave(3):  [A, A, A, B, B, B, C, C, C, D, D, D]  # interleaved
repeat(3):             [A, B, C, D, A, B, C, D, A, B, C, D]  # tiled
```

`repeat_interleave` e' quello corretto per GQA: garantisce che i 3 Q heads del gruppo 0 (indici 0,1,2) usino tutti lo stesso KV head 0.

### Schema dei Gruppi

```
Gruppo 0: Q_head[0], Q_head[1], Q_head[2]  ->  K_head[0], V_head[0]
Gruppo 1: Q_head[3], Q_head[4], Q_head[5]  ->  K_head[1], V_head[1]
Gruppo 2: Q_head[6], Q_head[7], Q_head[8]  ->  K_head[2], V_head[2]
Gruppo 3: Q_head[9], Q_head[10], Q_head[11] -> K_head[3], V_head[3]
```

Dopo `repeat_interleave(3)`, K e V hanno 12 'copie' (con ripetizioni),
abbinandosi correttamente ai 12 Q heads.


In [ ]:
# Capitolo 5 extra: Demo repeat_interleave
import torch

# Esempio con G=4 KV heads, n_rep=3
G, T, D = 4, 3, 2  # piccolo per leggibilita'
n_rep = 3

# KV heads: etichettati A,B,C,D per chiarezza
keys_gqa = torch.tensor([
    [[1.0, 0.1], [1.1, 0.2], [1.2, 0.3]],  # head 0 (A)
    [[2.0, 0.4], [2.1, 0.5], [2.2, 0.6]],  # head 1 (B)
    [[3.0, 0.7], [3.1, 0.8], [3.2, 0.9]],  # head 2 (C)
    [[4.0, 1.0], [4.1, 1.1], [4.2, 1.2]],  # head 3 (D)
]).unsqueeze(0)  # B=1, shape (1, G=4, T=3, D=2)

print(f'K prima di repeat_interleave: shape {keys_gqa.shape}')
for g in range(G):
    print(f'  KV head {g}: {keys_gqa[0,g,0].tolist()}')

keys_expanded = keys_gqa.repeat_interleave(n_rep, dim=1)
print(f'\nK dopo repeat_interleave(3, dim=1): shape {keys_expanded.shape}')
for h in range(G * n_rep):
    gruppo = h // n_rep
    print(f'  H head {h:2d} (gruppo {gruppo}): {keys_expanded[0,h,0].tolist()}')

print()
print('Verifica: gli H-head 0,1,2 usano tutti i dati di KV-head 0 (A)')
print('          gli H-head 3,4,5 usano tutti i dati di KV-head 1 (B)')
print('          ... etc.')


## 6.5 Confronto delle Funzioni di Attivazione

Ecco come le principali funzioni di attivazione si comportano per diversi valori di x:

| x | ReLU | GELU | SiLU | SwiGLU(x, 1) |
|---|------|------|------|---------------|
| -2 | 0 | -0.045 | -0.238 | -0.238 |
| -1 | 0 | -0.159 | -0.269 | -0.269 |
| 0  | 0 | 0      | 0      | 0      |
| 1  | 1 | 0.841  | 0.731  | 0.731  |
| 2  | 2 | 1.955  | 1.762  | 1.762  |

**Proprieta' chiave:**
- **ReLU**: zero per x<0, lineare per x>0. Non differenziabile in 0.
- **GELU**: smooth, ha valori leggermente negativi vicino a 0
- **SiLU**: smooth, self-gated (x * sigmoid(x)). Quasi identica a GELU per x > 0.
- **SwiGLU**: combina due segnali diversi, non e' una semplice funzione di un solo input

In pratica, SiLU e GELU sono molto simili. La vera differenza di SwiGLU e' il **meccanismo di gating** che permette alla rete di controllare selettivamente il flusso di informazione attraverso le dimensioni del FFN.


In [ ]:
# Capitolo 6 extra: Confronto dettagliato delle attivazioni
import torch
import torch.nn.functional as F
import math

x_vals = torch.tensor([-3., -2., -1., -0.5, 0., 0.5, 1., 2., 3.])

relu = F.relu(x_vals)
gelu = F.gelu(x_vals)
silu = F.silu(x_vals)

print('Confronto funzioni di attivazione:')
print(f'{"x":>6}  {"ReLU":>8}  {"GELU":>8}  {"SiLU":>8}')
print('-' * 38)
for i, xv in enumerate(x_vals):
    print(f'{xv.item():>6.1f}  {relu[i].item():>8.4f}  {gelu[i].item():>8.4f}  {silu[i].item():>8.4f}')

print()
print('Proprieta numeriche:')
print(f'  SiLU(0) = {F.silu(torch.tensor(0.0)).item():.4f} (neutro: no output)')
print(f'  SiLU(1) = {F.silu(torch.tensor(1.0)).item():.4f} (parzialmente attivato)')
print(f'  SiLU(5) = {F.silu(torch.tensor(5.0)).item():.4f} (quasi lineare)')
print(f'  SiLU(-5)= {F.silu(torch.tensor(-5.0)).item():.4f} (quasi zero)')

# SwiGLU demo: l'effetto del gate
print()
print('Demo SwiGLU: gate controlla up')
print('  gate_signal  up_signal  SiLU(gate)*up')
print('  -----------  ---------  -------------')
for gate, up in [(-2, 1), (-1, 1), (0, 1), (1, 1), (2, 1), (0, -1), (0, 2)]:
    result = F.silu(torch.tensor(float(gate))) * up
    print(f'  {gate:>11}  {up:>9}  {result.item():>13.4f}')


## 7.6 Analisi del Flusso del Gradiente

Il vantaggio delle residual connections diventa chiaro analizzando il backpropagation.

### Senza Residual (feed-forward puro)

```
output = sublayer_L(sublayer_{L-1}(...sublayer_1(x)...))
∂loss/∂x = ∂loss/∂output * prod_{l=1}^L J_l
```

dove J_l e' la Jacobiana del layer l. Il prodotto di molte Jacobiane tende a zero (vanishing) o infinito (exploding) con L grande.

### Con Residual (Pre-Norm style)

```
x_l = x_{l-1} + sublayer_l(norm(x_{l-1}))
∂x_l/∂x_{l-1} = I + J_l  (identita + Jacobiana)
∂loss/∂x = ∂loss/∂x_L * prod_l (I + J_l)
```

Il termine `I` garantisce che anche se `J_l ≈ 0` (sublayer non contribuisce), il gradiente fluisce comunque attraverso l'identita'.

**All'inizio del training** (pesi inizializzati casualmente con piccola norma):
- `J_l ≈ 0` (il sublayer contribuisce poco)
- `I + J_l ≈ I` (il gradiente fluisce invariato)
- Il modello e' efficacemente vicino all'identita'

**Dopo il training** (sublayer imparano trasformazioni significative):
- `J_l` non e' piu' piccola
- Il residual permette correzioni graduali rispetto all'identita'


In [ ]:
# Capitolo 7 extra: Visualizzazione del gradient flow con/senza residual
import torch
import torch.nn as nn

torch.manual_seed(0)
C = 32
L = 20

def make_block(use_residual=True):
    class Block(nn.Module):
        def __init__(self, res):
            super().__init__()
            self.linear = nn.Linear(C, C)
            self.norm = nn.LayerNorm(C)
            self.use_residual = res
        def forward(self, x):
            out = torch.relu(self.linear(self.norm(x)))
            if self.use_residual:
                return x + out
            return out
    return Block(use_residual)

# Due modelli: con e senza residual
model_res = nn.Sequential(*[make_block(True) for _ in range(L)])
model_nores = nn.Sequential(*[make_block(False) for _ in range(L)])

x = torch.randn(1, C, requires_grad=True)

# Calcola la norma del gradiente di ogni layer (con residual)
out_res = model_res(x)
loss_res = out_res.sum()
loss_res.backward()
grad_norm_res = x.grad.norm().item() if x.grad is not None else 0

x2 = torch.randn(1, C, requires_grad=True)
out_nores = model_nores(x2)
loss_nores = out_nores.sum()
loss_nores.backward()
grad_norm_nores = x2.grad.norm().item() if x2.grad is not None else 0

print(f'Norma del gradiente all input ({L} layer, C={C}):')
print(f'  Con residual:    {grad_norm_res:.6f}')
print(f'  Senza residual:  {grad_norm_nores:.6f}')
print()
print('Con residual: il gradiente fluisce meglio attraverso i layer')
print('Senza residual: il gradiente puo vanishing o exploding')
print('(i risultati variano per seed diversi, ma il trend e consistente)')


## 8.4 Weight Tying: Perche' Non e' Usato Qui

In GPT-2, i pesi di `tok_emb` e `out_head` sono **condivisi** (weight tying):

```python
# GPT-2 usa weight tying:
self.out_head.weight = self.tok_emb.weight
```

**Motivazione:** il token embedding e la proiezione vocab operano sullo stesso spazio di embedding — ha senso che lo stesso vettore che rappresenta il token 'cane' in input sia anche usato per calcolare il logit di 'cane' nell'output.

**Parametri risparmiati:** 50257 * 768 = **38.6M parametri**

**Perche' gpt_model4.py non la usa:**
- Scelta didattica: semplifica il codice
- In modelli grandi (LLaMA), weight tying non e' usato (le dimensioni del tokenizer e dell'embedding possono essere diverse)
- LLaMA 3: vocab=128256, emb=4096, quindi out_head ha 128256*4096 = 525M parametri

**Come aggiungere weight tying a gpt_model4.py:**
```python
class GPT4ModelTied(GPT4Model):
    def __init__(self, cfg):
        super().__init__(cfg)
        # Condividi i pesi dopo l'inizializzazione
        self.out_head.weight = self.tok_emb.weight
```


In [ ]:
# Capitolo 8 extra: Analisi del tok_emb e out_head
import torch
import torch.nn as nn

V, C = 50257, 768  # vocab_size, emb_dim

# Simulazione di tok_emb
tok_emb = nn.Embedding(V, C)
out_head = nn.Linear(C, V, bias=False)

print('tok_emb (nn.Embedding):')
print(f'  weight shape: {list(tok_emb.weight.shape)}')
print(f'  params: {tok_emb.weight.numel():,}')
print(f'  Dato token ID=100 -> embedding di shape {tok_emb(torch.tensor([100])).shape}')

print('\nout_head (nn.Linear, bias=False):')
print(f'  weight shape: {list(out_head.weight.shape)}')
print(f'  params: {out_head.weight.numel():,}')

# Dimostrazione: tok_emb lookup == out_head transposed
print('\nSe weight-tied: tok_emb.weight[i] == out_head.weight[i]')
print('=> il vettore embedding di ogni token e il suo logit vector sono lo stesso')

# Esempio di forward pass
torch.manual_seed(42)
idx = torch.randint(0, V, (2, 8))  # B=2, T=8
embedded = tok_emb(idx)
print(f'\nForward tok_emb: {idx.shape} -> {embedded.shape}')
print(f'  Max val: {embedded.max():.3f}, Min val: {embedded.min():.3f}')
print(f'  Std (attesa ~1 per init standard): {embedded.std():.3f}')

# Gradient sparsity
print()
print(f'Gradient sparsity di tok_emb:')
print(f'  Vocab size: {V:,}')
print(f'  Token unici in questo batch: {idx.unique().numel()}')
print(f'  => gradient non-zero solo per {idx.unique().numel()}/{V} righe')
print(f'  ({idx.unique().numel()/V*100:.3f}% di gradienti non-zero)')


In [ ]:
# Riepilogo finale: forward pass completo con shapes tracciate
import math, torch, torch.nn as nn, torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self,d,e=1e-5):
        super().__init__();self.e=e;self.s=nn.Parameter(torch.ones(d))
    def forward(self,x):
        return self.s*(x/torch.sqrt(x.pow(2).mean(-1,keepdim=True)+self.e))

def rope_freqs(hd,T,base=10000):
    t=1./(base**(torch.arange(0,hd,2).float()/hd))
    p=torch.arange(T,dtype=torch.float32)
    f=torch.outer(p,t)
    return torch.cos(f),torch.sin(f)

def apply_rope(x,c,s):
    _,_,T,D=x.shape;h=D//2
    x1,x2=x[...,:h],x[...,h:]
    c2=c[:T].unsqueeze(0).unsqueeze(0);s2=s[:T].unsqueeze(0).unsqueeze(0)
    return torch.cat([x1*c2-x2*s2,x2*c2+x1*s2],-1)

# Config ridotta per velocita'
B,T,C,H,G,L,V=2,16,128,4,2,3,1000
D=C//H
Hff=int(math.ceil(8/3*C/64)*64)

print('Forward pass con shapes (config ridotta per demo veloce):')
print(f'B={B}, T={T}, C={C}, H={H}, G={G}, L={L}')
print()

# tok_emb
idx = torch.randint(0, V, (B, T))
tok_emb = nn.Embedding(V, C)
x = tok_emb(idx)
print(f'tok_emb:      {list(idx.shape)} -> {list(x.shape)}')

# Blocco semplificato
c_buf, s_buf = rope_freqs(D, T)
norm1 = RMSNorm(C)
wq = nn.Linear(C, H*D, bias=False)
wk = nn.Linear(C, G*D, bias=False)
wv = nn.Linear(C, G*D, bias=False)
wo = nn.Linear(H*D, C, bias=False)

shortcut = x
x = norm1(x)
print(f'norm1:        {list(x.shape)}')
Q = wq(x).view(B,T,H,D).transpose(1,2)
K = wk(x).view(B,T,G,D).transpose(1,2)
V_t = wv(x).view(B,T,G,D).transpose(1,2)
print(f'Q shape:      {list(Q.shape)}  (B,H,T,D)')
print(f'K shape:      {list(K.shape)}  (B,G,T,D)')
Q = apply_rope(Q, c_buf, s_buf)
K = apply_rope(K, c_buf, s_buf)
print(f'Q after RoPE: {list(Q.shape)}')
K = K.repeat_interleave(H//G, 1)
V_t = V_t.repeat_interleave(H//G, 1)
print(f'K expanded:   {list(K.shape)}  (B,H,T,D)')
scores = Q @ K.transpose(2,3)
print(f'attn scores:  {list(scores.shape)}  (B,H,T,T)')
mask = torch.triu(torch.ones(T,T,dtype=torch.bool), diagonal=1)
scores.masked_fill_(mask, -torch.inf)
w = torch.softmax(scores/D**0.5, -1)
print(f'attn weights: {list(w.shape)}')
ctx = (w@V_t).transpose(1,2).reshape(B,T,C)
x = wo(ctx) + shortcut
print(f'output block: {list(x.shape)}')

# Final
final_norm = RMSNorm(C)
out_head_proj = nn.Linear(C, V, bias=False)
logits = out_head_proj(final_norm(x))
print(f'final logits: {list(logits.shape)}')
print()
print('Shape invariante B,T,C attraverso tutto il modello!')
print('Solo out_head porta da C a V (=vocab_size)')


## Nota sul codice sorgente: Commenti riga per riga

Una caratteristica distintiva di `gpt_model4.py` e' la presenza di commenti dettagliati su ogni riga, con le shape dei tensori.

Questo stile di documentazione e' fondamentale per l'apprendimento:

```python
# Esempio: forward di GroupedQueryAttention
queries = self.W_query(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
# Q: (B,T,C) -> (B,T,H,D) -> (B,H,T,D) = (2,12,256,64)

keys = self.W_key(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
# K: (B,T,C) -> (B,T,G,D) -> (B,G,T,D) = (2,4,256,64)
```

Il commento specifica:
1. **La variabile**: `Q`, `K`, `V`
2. **La trasformazione**: `(B,T,C) -> (B,T,H,D) -> (B,H,T,D)`
3. **I valori concreti**: `(2,12,256,64)` con la config default

Questo permette di seguire il flusso dei dati attraverso il codice senza dover tenere tutto nella testa mentalmente.

### Convenzioni nel file

| Lettera | Significato |
|---------|-------------|
| B | Batch size |
| T | Sequence length (time) |
| C | Embedding dimension (channels) |
| H | Number of query heads |
| G | Number of KV heads |
| D | Head dimension (= C/H) |
| V | Vocabulary size |
| H_ff | FFN hidden dimension |


In [ ]:
# Riepilogo finale: verifica dell'implementazione
# Questo blocco verifica che le shape siano corrette per la config small
import math

GPT4_CONFIG_SMALL = {
    'vocab_size': 50257,
    'context_length': 1024,
    'n_layers': 12,
    'n_heads': 12,
    'n_kv_heads': 4,
    'emb_dim': 768,
    'drop_rate': 0.1,
    'qkv_bias': False,
    'rope_base': 10000,
}

cfg = GPT4_CONFIG_SMALL
V = cfg['vocab_size']
T = cfg['context_length']
C = cfg['emb_dim']
H = cfg['n_heads']
G = cfg['n_kv_heads']
L = cfg['n_layers']
D = C // H
H_ff = int(math.ceil(8/3*C/256)*256)

print('Verifica shape per GPT4_CONFIG_SMALL (B=2, T_input=32):')
print('=' * 60)
B_test, T_test = 2, 32

checks = [
    ('in_idx',        (B_test, T_test)),
    ('tok_embeds',    (B_test, T_test, C)),
    ('dopo norm1',    (B_test, T_test, C)),
    ('Q (att)',       (B_test, H, T_test, D)),
    ('K (att, GQA)', (B_test, G, T_test, D)),
    ('K (expanded)', (B_test, H, T_test, D)),
    ('attn scores',  (B_test, H, T_test, T_test)),
    ('attn output',  (B_test, T_test, C)),
    ('dopo att+res', (B_test, T_test, C)),
    ('dopo norm2',   (B_test, T_test, C)),
    ('FFN gate',     (B_test, T_test, H_ff)),
    ('FFN output',   (B_test, T_test, C)),
    ('dopo L blocchi',(B_test, T_test, C)),
    ('final_norm',   (B_test, T_test, C)),
    ('logits',       (B_test, T_test, V)),
]

for name, shape in checks:
    print(f'  {name:<22} {str(shape)}')

print()
print('Tutti i tensori interni (tranne logits) hanno dimensione embedding C=768')
print('I logits hanno la dimensione del vocabolario V=50257')
